**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/catalystneuro/encore", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/huszar-2022"], check=True)
    os.chdir("/content/encore/entries/huszar-2022")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Huszár et al. (2022) from DANDI:000552

Paper. Huszár R, Zhang Y, Blockus H, Buzsáki G (2022). Preconfigured dynamics in the hippocampus are
guided by embryonic birthdate and rate of neurogenesis. *Nature Neuroscience* 25, 1201-1212.
doi:10.1038/s41593-022-01138-x.

Claims. `claims/huszar-2022.yaml`, written from the author manuscript in PubMed Central (PMC10807234)
with its Reporting Summary and Supplementary Information (Supplementary Tables 1 to 5), and the paper's code
(github.com/rhuszar/optotagCA1_final, commit 51320d4). Method details the claims file lacks are taken from
the Methods and legends of that text, cited beside the code in `huszar.py`.

Dataset. DANDI:000552, version `0.261113.1035` (doi:10.48324/dandi.000552/0.261113.1035), the most recent
published version. The paper cites no version (it was published before the dandiset). The earlier published
version, 0.230630.2304, lacks what the claims need: its release notes say it had no light-responsive labels,
no light pulses, units for only some sessions and LFP in only 13 files, while 0.261113.1035 holds all of
these for the 104 sessions. It therefore also matches the claims file's description better. Only the 104
processed files are read, streamed from the archive; the 11 raw wide-band files are not.

This notebook streams every processed file and takes about 47 minutes, most of it reading two LFP channels
per session for the theta analyses.

| Claim | Tested here |
|---|---|
| cohort-birthdates, st1-animal-sessions, sessions-durations, spwr-duration, unit-count, ed2-light-responsive-count | yes |
| fig1e-labelled-fraction, fig1f-rem-shift, fig1g-rate-by-birthdate, fig1h-burst-by-birthdate | yes |
| fig2b-synchrony-per-animal, fig2c-theta-cofiring, fig2d-theta-depth, fig2f-ripple-firing-by-birthdate, fig2g-ripple-cofiring | yes |
| maze-cohort, maze-task, novel-explorations | yes |
| fig5b-spatial-info-by-birthdate, fig5c-sbd-ratemap-overlap, fig5d-overlap-by-distance, fig5e-pv-decorrelation, fig5f-remap-together | yes |
| fig6c-theta-rate-preserved, fig6d-rate-lower-novel, fig6d-info-lower-novel, fig6c-novel-ratemap-corr-higher, fig6e-sbd-novel-overlap, fig6f-remap-together-novel | yes, open fields at an assumed scale |
| fig7b-sbd-convergence, fig7ce-convergence-by-birthdate, fig7d-transmission-gamma, fig7fg-convergence-cofiring | yes |
| ed1-brdu-specificity, fig1c-radial-depth | blocked: histology not in the dataset |
| ed2-waveform-validation | blocked: light-evoked waveforms need the wide-band signal, absent for 91 of 96 sessions |
| ed3-depth-control | yes |
| ed5-matched-controls | blocked: no L-ratio in the dataset (the rate-matched half is run for the record) |
| fig3c, fig3e, ed6, fig4d-model-fit | blocked: not run; the assembly matrix as first planned (dense, 1 ms bins) needs 9 to 12 GB per session, and a chunked version was not built in the time (see the attempt record) |
| ed7-sbd-coactivation | blocked: depends on fig3e-assembly-expression (part e is run for the record) |
| ed10-waveform-matched-controls | blocked: depends on ed2-waveform-validation (its own parts are run for the record) |

In [ ]:
import json, time, zlib, resource, collections
import numpy as np, pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import huszar as h
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
KEY = "huszar-2022"
def seed(claim): return zlib.crc32(f"{KEY}/{claim}".encode())
SEEDS_PLUS = [0, 1000, 2000, 3000, 4000]
T0 = time.time()

## Data check

In [ ]:
metas = h.survey(workers=8)
S = pd.DataFrame([{"session": m["session_id"], "subject": m["subject_id"], "sex": m["sex"], "age": m["age"],
                   "birthdate": m["birthdate"], "units": m["n_units"],
                   "pyr": sum(c == "Pyramidal Cell" for c in m.get("u_putative_cell_type", [])),
                   "narrow": sum(c == "Narrow Interneuron" for c in m.get("u_putative_cell_type", [])),
                   "wide": sum(c == "Wide Interneuron" for c in m.get("u_putative_cell_type", [])),
                   "unknown_type": sum(c not in ("Pyramidal Cell", "Narrow Interneuron", "Wide Interneuron")
                                       for c in m.get("u_putative_cell_type", [])),
                   "light_resp": int(np.sum(m.get("u_light_responsive", []))),
                   "lfp_min": m["lfp_duration_s"] / 60, "last_spike_min": m["last_spike"] / 60,
                   "lfp_rate": m.get("lfp_rate"), "ripples": len(m["rip_start"]),
                   "states": "/".join(sorted(set(m["sleep"]["label"]))) if m["sleep"] else "",
                   "pulses": m["n_pulses"], "beh_epochs": len(h.behaviour_epochs(m)),
                   "envs": ",".join(e[2] for e in h.behaviour_epochs(m)),
                   "trials": len(m["trials"]["start_time"]) if m["trials"] else 0}
                  for m in metas]).sort_values("session").reset_index(drop=True)
META = {m["session_id"]: m for m in metas}
S["homecage_mouse"] = ~S.subject.isin(h.HOMECAGE_EXCLUDED)
print(len(S), "processed files,", S.subject.nunique(), "subjects,", S.session.nunique(), "distinct session ids")
print("LFP rate(s):", S.lfp_rate.unique(), "Hz; LFP duration against last spike, max |difference| =",
      round(float((S.lfp_min - S.last_spike_min).abs().max()), 2), "min")
print(S.groupby("subject").agg(sex=("sex", "first"), birthdate=("birthdate", "first"), sessions=("session", "size"),
                               units=("units", "sum"), pyr=("pyr", "sum"), light_resp=("light_resp", "sum")).to_string())
print("\ncell types over all files:", S[["pyr", "narrow", "wide", "unknown_type"]].sum().to_dict())
print("sleep-state labels per session:", S.states.value_counts().to_dict())
print("behavioural environments:", collections.Counter(e for s in S.envs for e in s.split(",") if e))
print("ages:", S.age.unique(), " sexes:", S.sex.unique())
# deposited twice? identical spike counts or ripple counts between sessions
dup = S[S.duplicated(["units", "ripples", "pulses"], keep=False)]
print("sessions sharing unit, ripple and pulse counts (possible duplicates):", len(dup))
cols = [set(m["u_columns"]) for m in metas]
print("units table columns in every file:", sorted(set.intersection(*cols)), "; in some files only:", sorted(set.union(*cols) - set.intersection(*cols)))
print("unit columns naming an isolation metric (L-ratio, isolation distance):", [c for c in set.union(*cols) if "ratio" in c.lower() or "isol" in c.lower()])
print("waveform_mean shape after the unit axis:", collections.Counter(tuple(m["wf_shape"][1:]) for m in metas if m["wf_shape"]),
      "; electrodes per file:", collections.Counter(m["n_electrodes"] for m in metas))
print("files with electrodes on two faces of the shanks (probe_side Front and Back), by mouse:", dict(collections.Counter(m["subject_id"] for m in metas if len(m["probe_sides"]) > 1)))

In [ ]:
# Clock checks (protocol, author step 2): spikes against light pulses, ripples and position, each with a shifted control.
chk_sessions = [s for s in ["e13_16f1_210302", "e14_2m3_201125", "e15_9f1_200720", "e16_3m1_210201", "e15_4m2_191213"]]
CK = pd.DataFrame(h.run_sessions(h.clock_checks, [META[s] for s in chk_sessions]))
print(CK.round(3).to_string())

The files describe 17 mice and 104 sessions, one file per session with no duplicates by unit, ripple and pulse
counts. Every session has units with cell types, light pulses, ripples and LFP at 1,250 Hz; 101 sessions have
wake, non-REM and REM states, one has no REM, and two (e13_1f1_200305 and e13_1f1_200307) have wake only. In
the five sessions checked, light-responsive cells fire 2.7 to 15 times more in the 10 ms after the pulse onsets
than 1 s later, pyramidal cells fire 4.5 to 6.3 times faster in the ripples than in windows shifted by 1 s, and in
the four maze sessions among them the median spatial information falls from between 0.56 and 1.07 bits/s to between
0.13 and 0.19 bits/s when spikes are shifted by 30 s against the position. Spikes, pulses, ripples and position are
therefore consistent with one clock in those sessions; other sessions were not checked this way. The theta
section adds the same contrast for the LFP.

The units table has the same 21 columns in every file and none is an isolation metric (no L-ratio). `waveform_mean`
holds each unit's mean waveform on every channel (72 samples by 64 or 128 channels). The 25 files with 128 channels
(9 mice) mark their electrodes Front or Back (`probe_side`): two faces of the same four shanks, with the same x
positions on both faces. The two wake-only sessions hold a single sleep-state row each, Awake over the whole recording
(414.7 and 331.5 min), with 1,942 and 1,247 ripples detected, so they were scored but carry no sleep.

## Counts: cohort, sessions, units

> Claim cohort-birthdates: Chronic recordings were made from 17 adult mice (12 males, 5 females) electroporated
> at E13.5 (n = 4), E14.5 (n = 3), E15.5 (n = 6) and E16.5 (n = 4).

In [ ]:
sub = S.groupby("subject").agg(sex=("sex", "first"), birthdate=("birthdate", "first")).reset_index()
print("mice:", len(sub), " males:", int((sub.sex == "M").sum()), " females:", int((sub.sex == "F").sum()))
print("per birthdate:", sub.birthdate.value_counts().sort_index().to_dict())
st1 = pd.DataFrame({"subject": h.ST1_ORDER})
st1["table_birthdate"] = [h.birthdate(s) for s in h.ST1_ORDER]
st1 = st1.merge(sub, on="subject", how="left")
print("subjects of Supplementary Table 1 found in the dataset:", int(st1.sex.notna().sum()), "of", len(st1),
      "; birthdate agrees:", int((st1.table_birthdate == st1.birthdate).sum()))

**Result: reproduced.** The files hold 17 mice, 12 male and 5 female, with 4, 3, 6 and 4 mice electroporated at E13.5 to E16.5; all 17 animals of Supplementary Table 1 are present with the table's birthdate. Sex and birthdate come from the subject ids, as the files say.

> Claim st1-animal-sessions: Supplementary Table 1 lists each mouse's recording sessions and light-responsive
> cells; the 15 homecage mice contribute 96 sessions and 664 light-responsive cells, and the 17 mice 104 sessions.

In [ ]:
per = S.groupby("subject").agg(sessions=("session", "size"), light_resp=("light_resp", "sum"), pyr=("pyr", "sum"))
T = pd.DataFrame({"subject": h.ST1_ORDER, "table_sessions": h.ST1_SESSIONS, "table_light": h.ST1_LIGHT}).set_index("subject")
T = T.join(per)
T["pct_light"] = (100 * T.light_resp / T.pyr).round(2)
T["sessions_match"] = T.sessions == T.table_sessions
T["light_match"] = T.light_resp == T.table_light
print(T.to_string())
hm = T.index[~T.index.isin(h.HOMECAGE_EXCLUDED)]
print("\nsessions of the 15 homecage mice:", int(T.loc[hm, "sessions"].sum()), "; of all 17:", int(T.sessions.sum()))
print("light-responsive cells of the 15 homecage mice:", int(T.loc[hm, "light_resp"].sum()))
print("mice whose session count matches:", int(T.sessions_match.sum()), "of 17; light-responsive count matches:",
      int(T.light_match.sum()), "of 17")

**Result: reproduced.** Every one of the 17 mice has the table's session count and the table's number of light-responsive cells; the 15 homecage mice give 96 sessions and 664 light-responsive cells, and the 17 mice 104 sessions.

> Claim sessions-durations: The recordings comprise 92 sessions lasting 113.7 to 475.6 min (median 309.1 min),
> each including a homecage period of sleep and wake.

In [ ]:
S["duration_min"] = S.lfp_min
S["has_nrem"] = S.states.str.contains("Non-REM")
for lab, sel in [("96 sessions of the 15 homecage mice", S.homecage_mouse), ("104 sessions of all 17 mice", S.index >= 0)]:
    d = S.loc[sel, "duration_min"]
    print(f"{lab}: n = {len(d)}, median {d.median():.1f} min, range {d.min():.1f} to {d.max():.1f} min; "
          f"all within 102.3-523.2: {bool(((d >= 102.33) & (d <= 523.16)).all())}; "
          f"sessions without non-REM: {int((~S.loc[sel, 'has_nrem']).sum())}")
print("sessions without non-REM sleep:", S.loc[~S.has_nrem, ["session", "states"]].values.tolist())
print("count reading of 92 sessions: none of the session sets has 92; 96 is within 92 +- 9.2:", abs(96 - 92) <= 9.2,
      "; 104 is:", abs(104 - 92) <= 9.2)
for s in ["e13_1f1_200305", "e13_1f1_200307"]:
    ss = pd.DataFrame(META[s]["sleep"]); ss["min"] = (ss.stop_time - ss.start_time) / 60
    print(f"{s}: recording {S.set_index('session').lfp_min[s]:.1f} min; sleep-state rows {len(ss)}, labels {ss.label.value_counts().to_dict()}, "
          f"scored {ss['min'].sum():.1f} min from {ss.start_time.min()/60:.1f} to {ss.stop_time.max()/60:.1f} min; ripples {len(META[s]['rip_start'])}")

**Result: not reproduced.** No session set has 92 sessions; under the 96-session reading the count (96) is within 92 +- 9.2, the median is 309.9 min and every duration lies within 113.7 to 475.6 min, the paper's exact range. Two sessions (e13_1f1_200305, e13_1f1_200307) hold only wake in their sleep states (one Awake row over the whole recording), so the criterion's requirement that every session holds non-REM sleep fails, and the criterion has no partial outcome for this case; this is asked of a reader (questions/huszar-2022.yaml).

> Claim unit-count: The dataset includes 9,471 well-isolated units, 7,411 putative pyramidal cells and 2,060
> putative interneurons (1,556 narrow waveform, 504 wide waveform).

In [ ]:
for lab, sel in [("(a) 96 sessions of the 15 homecage mice", S.homecage_mouse), ("(b) 104 sessions of all 17 mice", S.index >= 0)]:
    x = S.loc[sel]
    print(f"{lab}: units {x.units.sum()}, pyramidal {x.pyr.sum()}, interneurons {x.narrow.sum() + x.wide.sum()} "
          f"(narrow {x.narrow.sum()}, wide {x.wide.sum()}), other type {x.unknown_type.sum()}")

**Result: reproduced.** Over the 96 sessions of the 15 homecage mice there are 7,411 pyramidal cells and 2,060 interneurons (1,556 narrow, 504 wide), the paper's numbers exactly; the total is 9,472 units, the paper's 9,471 plus one unit without a cell type.

> Claim ed2-light-responsive-count: 664 of 7,411 putative pyramidal neurons (8.96%) were light-responsive, that
> is, birthdated, leaving 6,747 nonresponsive.

In [ ]:
x = S[S.homecage_mouse]
U = pd.DataFrame([{"session": m["session_id"], "subject": m["subject_id"], "ct": c, "lr": l, "p_salt": p, "h_rel": r}
                  for m in metas for c, l, p, r in zip(m["u_putative_cell_type"], m["u_light_responsive"],
                                                       m["u_opto_p_salt"], m["u_opto_h_reliable"])])
U = U[~U.subject.isin(h.HOMECAGE_EXCLUDED)]
pyrU = U[U.ct == "Pyramidal Cell"]
a = int(pyrU.lr.sum()); b = int(((pyrU.p_salt <= 0.001) & (pyrU.h_rel.astype(bool))).sum())
b2 = int(((pyrU.p_salt < 0.001) & (pyrU.h_rel.astype(bool))).sum())
print(f"(a) stored light_responsive: {a} of {len(pyrU)} pyramidal ({100*a/len(pyrU):.2f}%), nonresponsive {len(pyrU)-a}")
print(f"(b) from the stored SALT p and reliability flag (p_salt <= 0.001 and h_reliable): {b}; with p < 0.001 (the code's): {b2}")
print("light_responsive outside pyramidal cells:", int(U[U.ct != 'Pyramidal Cell'].lr.sum()))
print("p_salt missing among pyramidal cells:", int(pyrU.p_salt.isna().sum()))

**Result: reproduced.** The stored labels give 664 light-responsive cells of 7,411 pyramidal cells (8.96%), and the same 664 follow from the stored SALT p and reliability flag.

> Claim spwr-duration: The median duration of SPW-Rs across all recordings is 36.4 ms, under detection limits of
> 20 to 200 ms.

In [ ]:
rows = []
for m in metas:
    d = np.asarray(m["rip_dur"]) * 1000
    hc = h.homecage_intervals(m)
    rip = np.c_[m["rip_start"], m["rip_stop"]]
    k = h.in_intervals(rip[:, 0], hc) & h.in_intervals(rip[:, 1], hc)
    rows.append({"session": m["session_id"], "subject": m["subject_id"], "all": d, "hc": d[k]})
R = pd.DataFrame(rows)
for setlab, sel in [("96 homecage-mouse sessions", ~R.subject.isin(h.HOMECAGE_EXCLUDED)), ("104 sessions", R.index >= 0)]:
    for rl in ["all", "hc"]:
        arrs = R.loc[sel, rl]
        pooled = np.concatenate(arrs.values)
        meds = np.array([np.median(a) for a in arrs if len(a)])
        print(f"{setlab}, ripples={'(i) all detected' if rl=='all' else '(ii) homecage'}: n = {len(pooled)}; "
              f"(a) pooled median {np.median(pooled):.2f} ms; (b) median of session medians {np.median(meds):.2f}; "
              f"(c) mean of session medians {np.mean(meds):.2f}; range {pooled.min():.1f} to {pooled.max():.1f} ms; "
              f"outside 20-200 ms: {int(((pooled < 20) | (pooled > 200)).sum())}")
allr = np.concatenate(R["all"].values)
print("ripple durations: start-stop agree with ripple_durations column to", 
      np.max(np.abs(np.concatenate([(np.asarray(m['rip_stop']) - np.asarray(m['rip_start'])) * 1000 - np.asarray(m['rip_dur']) * 1000 for m in metas]))), "ms")

**Result: reproduced.** Every combination of reading gives a median between 36.00 and 36.80 ms, within 36.4 +- 3.64 ms; the mean of session medians of all detected ripples (the code's reading) is 36.28 ms over the 96 sessions. All 494,578 ripples last 20.0 to 199.2 ms.

> Claim fig1e-labelled-fraction: The fraction of light-responsive pyramidal cells per session follows a
> bell-shaped wave over birthdates, peaking at E15.5 (E13.5 5.39%, E14.5 11.16%, E15.5 14.74%, E16.5 9.16%).

In [ ]:
x = S[S.homecage_mouse].copy()
x["pct"] = 100 * x.light_resp / x.pyr
g = x.groupby("birthdate").pct
print(pd.DataFrame({"sessions": g.size(), "mean": g.mean().round(2), "sd": g.std().round(2)}).to_string())
groups = [x[x.birthdate == b].pct.values for b in h.BIRTHDATES]
for k in SEEDS_PLUS:
    F, p = h.boot_anova(groups, rng=np.random.default_rng(seed("fig1e-labelled-fraction") + k))
    print(f"seed +{k}: bootstrap ANOVA F = {F:.3f}, p = {p:.4f}")
print("by mouse:", x.groupby(["birthdate", "subject"]).pct.mean().round(2).to_dict())

**Result: reproduced.** The per-session means are 5.39, 11.16, 14.74 and 9.16% over 30, 24, 24 and 18 sessions, the paper's values, with F = 8.525 as printed; the bootstrap ANOVA gives p < 0.0002 under all five seeds, E13.5 is lowest and E15.5 highest.

## Homecage single-cell and SPW-R measures

One pass over the 96 sessions of the 15 homecage mice (`h.homecage_session`). The homecage period is the
recording up to the second behavioural subsession, without the light pulse periods and the maze epochs
(claims, opening comment; getHomecageMetrics.m). SPW-Rs are the dataset's (bz_FindRipples, as the Methods),
those starting and ending in the homecage period.

In [ ]:
HM = [META[s] for s in S.loc[S.homecage_mouse, "session"]]
t = time.time()
hc_res = h.run_sessions(h.homecage_session, HM)
CELLS = pd.DataFrame([c for r in hc_res for c in r[0]])
PAIRS = pd.DataFrame([p for r in hc_res for p in r[1]])
HCI = pd.DataFrame([r[2] for r in hc_res])
LR = CELLS[CELLS.lr]
print(f"{time.time()-t:.0f} s; pyramidal cells {len(CELLS)}, light-responsive {len(LR)}; pairs SBD {int(PAIRS.sbd.sum())}, DBD {int((~PAIRS.sbd).sum())}")
print("homecage period per session (min): median", round(HCI.hc_dur.median() / 60, 1), "range", round(HCI.hc_dur.min() / 60, 1), "-", round(HCI.hc_dur.max() / 60, 1))
print("homecage ripples per session: median", int(HCI.n_rip_hc.median()), "range", HCI.n_rip_hc.min(), "-", HCI.n_rip_hc.max(), "; of all ripples", round(HCI.n_rip_hc.sum() / HCI.n_rip.sum(), 3))
print("light-responsive cells per birthdate:", LR.groupby("birthdate").size().to_dict())

In [ ]:
def by_bd(df, col, claim, test="kw", n_seeds=5):
    groups = [df[df.birthdate == b][col].dropna().values for b in h.BIRTHDATES]
    med = [float(np.median(g)) for g in groups]
    print(f"{col}: n = {[len(g) for g in groups]}, medians = {np.round(med, 4).tolist()}, ranges = "
          f"{[(round(float(g.min()), 3), round(float(g.max()), 3)) for g in groups]}")
    ps = []
    for k in SEEDS_PLUS[:n_seeds]:
        f = h.boot_kw if test == "kw" else h.boot_anova
        st, p = f(groups, rng=np.random.default_rng(seed(claim) + k))
        ps.append(p)
    print(f"  bootstrap {'Kruskal-Wallis H' if test=='kw' else 'ANOVA F'} = {st:.3f}; p under the claim's seed and +1000..+4000: {ps}")
    return med, ps

> Claim fig1g-rate-by-birthdate: Earlier-born light-responsive pyramidal neurons have higher firing rates
> (medians 1.57, 1.63, 1.62 and 1.16 Hz for E13.5 to E16.5).

In [ ]:
med, ps = by_bd(LR, "rate", "fig1g-rate-by-birthdate")
print("E16.5 lowest:", int(np.argmin(med)) == 3)
print("within tolerance:", [abs(a - b) <= t for a, b, t in zip(med, [1.57, 1.63, 1.62, 1.16], [0.157, 0.163, 0.162, 0.116])])
print("by mouse (median rate):", LR.groupby(["birthdate", "subject"]).rate.median().round(2).to_dict())

**Result: reproduced.** Medians 1.52, 1.62, 1.54 and 1.15 Hz over 184, 132, 233 and 115 light-responsive cells are within tolerance, E16.5 is lowest, and the bootstrap Kruskal-Wallis gives p = 0.034 (0.034 to 0.041 over the five seeds).

> Claim fig1h-burst-by-birthdate: Burst index is an inverted-U function of birthdate (medians 0.1167, 0.1272,
> 0.136, 0.1082 for E13.5 to E16.5).

In [ ]:
med, ps = by_bd(LR, "burst", "fig1h-burst-by-birthdate")
print("legend 2-5 ms: inverted-U:", h.inverted_u(med), "; within tolerance:",
      [abs(a - b) <= t for a, b, t in zip(med, [0.1167, 0.1272, 0.136, 0.1082], [0.0117, 0.0127, 0.0136, 0.0108])])
med35, ps35 = by_bd(LR, "burst35", "fig1h-burst-by-birthdate", n_seeds=1)
print("code 3-5 ms variant: inverted-U:", h.inverted_u(med35))
print("by mouse (median 2-5 ms):", LR.groupby(["birthdate", "subject"]).burst.median().round(3).to_dict())

**Result: partially reproduced.** The bootstrap Kruskal-Wallis gives p <= 0.0004 under all seeds and the medians form an inverted U (0.087, 0.091, 0.103, 0.080, highest at E15.5), but all four medians are about 25% below the paper's (0.117 to 0.136) and outside tolerance; the code's 3-5 ms variant is lower still (0.074 to 0.094).

> Claim fig2f-ripple-firing-by-birthdate: SPW-R firing rates (medians 7.8, 8.75, 8.01, 6.13 Hz) and
> participation (0.245, 0.268, 0.235, 0.194) of light-responsive neurons are inverted-U functions of birthdate.

In [ ]:
m1, p1 = by_bd(LR, "rip_rate", "fig2f-ripple-firing-by-birthdate")
m2, p2 = by_bd(LR, "rip_part", "fig2f-ripple-firing-by-birthdate")
print("rate: inverted-U", h.inverted_u(m1), "within tolerance", [abs(a - b) <= 0.1 * b for a, b in zip(m1, [7.8, 8.75, 8.01, 6.13])])
print("participation: inverted-U", h.inverted_u(m2), "within tolerance", [abs(a - b) <= 0.1 * b + 1e-12 for a, b in zip(m2, [0.245, 0.268, 0.235, 0.194])])
print("by mouse (median SPW-R rate):", LR.groupby(["birthdate", "subject"]).rip_rate.median().round(2).to_dict())

**Result: reproduced.** SPW-R rate medians 7.48, 8.82, 7.91 and 6.13 Hz and participation 0.237, 0.269, 0.236 and 0.194 are within tolerance, both are inverted U, and both bootstrap Kruskal-Wallis tests give p <= 0.001 under all seeds.

> Claim fig2g-ripple-cofiring: SBD pairs cofire more than DBD pairs in SPW-Rs (median 0.0399 against 0.0231).

In [ ]:
def sbd_dbd(df, col):
    s, d = df[df.sbd][col].dropna(), df[~df.sbd][col].dropna()
    p = stats.ranksums(s, d).pvalue
    print(f"{col}: SBD median {s.median():.4f} (n = {len(s)}), DBD median {d.median():.4f} (n = {len(d)}), "
          f"two-sided Wilcoxon rank-sum p = {p:.3g}; dropped (NaN) SBD {int(df[df.sbd][col].isna().sum())}, DBD {int(df[~df.sbd][col].isna().sum())}")
    return s.median(), d.median(), p
s, d, p = sbd_dbd(PAIRS, "rip_corr")
print("within tolerance:", abs(s - 0.0399) <= 0.004, abs(d - 0.0231) <= 0.0023)
print("per birthdate SBD/DBD medians:", PAIRS.groupby(["birthdate", "sbd"]).rip_corr.median().round(4).to_dict())

**Result: reproduced.** SBD median 0.0385 against DBD 0.0226 (3,738 and 50,171 pairs with a correlation), rank-sum p = 4.7e-98; both within tolerance.

> Claim fig2b-synchrony-per-animal: Same-birthdate (SBD) pairs fire more synchronously than different-birthdate
> (DBD) pairs in 11 of 13 animals.

In [ ]:
rows = []
for sbj, g in PAIRS.groupby("subject"):
    s, d = g[g.sbd], g[~g.sbd]
    ra = (s.rip_corr.dropna(), d.rip_corr.dropna())
    rb = (s["sync"].dropna(), d["sync"].dropna())
    pa = stats.ranksums(*ra).pvalue if len(ra[0]) and len(ra[1]) else np.nan
    rows.append({"subject": sbj, "sbd_pairs": len(s), "dbd_pairs": len(d),
                 "a_sbd_med": ra[0].median() if len(ra[0]) else np.nan, "a_dbd_med": ra[1].median(), "a_p": pa,
                 "a_higher_sig": bool(len(ra[0]) and ra[0].median() > ra[1].median() and pa < 0.05),
                 "b_diff": (rb[0].mean() - rb[1].mean()) if len(rb[0]) else np.nan})
A = pd.DataFrame(rows)
A["b_positive"] = A.b_diff > 0
print(A.round(4).to_string())
el = A[A.sbd_pairs > 0]
t13 = A[~A.subject.isin(["e13_1f1", "e13_1m1"])]
print(f"the 13 animals of the claim (without e13_1f1 and e13_1m1, the two E13.5 mice with one light-responsive cell in most sessions): "
      f"(a) SBD higher with p < 0.05 in {int(t13.a_higher_sig.sum())}; (b) z-scored CCG difference > 0 in {int(t13.b_positive.sum())}")
print(f"animals with SBD pairs: {len(el)}; (a) SBD higher with p < 0.05: {int(el.a_higher_sig.sum())}; "
      f"(a') SBD median higher regardless of p: {int((el.a_sbd_med > el.a_dbd_med).sum())}; (b) mean z-scored CCG difference > 0: {int(el.b_positive.sum())}")

**Result: reproduced.** Of the 13 animals of the claim, SBD pairs cofire more in SPW-Rs than DBD pairs with p < 0.05 in 10 (reading a), and the z-scored CCG of SBD pairs exceeds that of DBD pairs in 12 (reading b); both are in the range 10 to 12. The two E13.5 mice the paper leaves out do have SBD pairs in the files (1 and 2 pairs).

## Theta

Two LFP channels per session are read in full (`h.theta_session`), over the 96 homecage-mouse sessions and the
8 sessions of the two novel-only mice. The main one follows the Methods' rule, a channel with a positive sharp wave
(above the centre of the pyramidal layer). The sharp wave of each channel is its mean LFP 0 to 50 ms after the
ripple peaks minus its mean 100 to 50 ms before them, in microvolts, over the ripples of homecage non-REM sleep (all
ripples in the sessions with 20 or fewer of them, printed below). Among the channels of the centre's shank with a
positive sharp wave and larger rel_y than the centre (above it), the one with the largest ripple power is taken, on
another shank when that shank has none, and the channel next to the centre with larger rel_y when no shank has one
(assumption, `pick_theta_channel`; how the paper picked among qualifying channels is not said). Both halves of the
Methods' phrase are required: in the first revision a positive channel below the centre also qualified, which 6
sessions used (verifier, round 2). The second channel is the pyramidal layer centre itself, run as a variant, and
each theta claim is also shown on the sessions whose channel follows the rule.

In [ ]:
TH_SESS = [META[s] for s in S.loc[S.homecage_mouse | S.subject.isin(h.NOVEL_MICE), "session"]]
t = time.time()
th_res = h.run_sessions(h.theta_session_novel, TH_SESS, workers=5)
TC_ALL = pd.DataFrame([c for r in th_res for c in r[0]])
TP_ALL = pd.DataFrame([p for r in th_res for p in r[1]])
TO = pd.DataFrame([{k: v for k, v in r[2].items() if k != "theta_corr_mat"} for r in th_res])
TR_ALL = pd.DataFrame([x for r in th_res for x in r[3]])
TMAT = {r[2]["session"]: r[2]["theta_corr_mat"] for r in th_res}
# main reading: the theta channel of the Methods' rule; variant: the pyramidal layer centre
TC, TP, TR = (d[d.ref == "rule"].copy() for d in (TC_ALL, TP_ALL, TR_ALL))
TCc, TPc, TRc = (d[d.ref == "centre"].copy() for d in (TC_ALL, TP_ALL, TR_ALL))
RULE_SESS = set(TO[TO["where"] != "none positive, next channel with larger rel_y"].session)
print(f"{time.time()-t:.0f} s, {len(th_res)} sessions; peak memory of this process {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1e6:.2f} GB")
print("theta channel, by where a channel with a positive sharp wave was found:", TO["where"].value_counts().to_dict())
print("theta channel position relative to the layer centre of its shank (rel_y difference, um):", TO.dy_theta.value_counts().sort_index().to_dict())
print("sharp wave (uV, 0-50 ms after ripple peaks minus 100-50 ms before): theta channel median", round(TO.sw_theta.median(), 1),
      "range", round(TO.sw_theta.min(), 1), "to", round(TO.sw_theta.max(), 1), "; pyramidal layer centre median", round(TO.sw_pyr.median(), 1))
print("on the centre's shank, channel next above the centre: median", round(TO.sw_next_above.median(), 1), "uV, positive in", int((TO.sw_next_above > 0).sum()),
      "of", int(TO.sw_next_above.notna().sum()), "; next below: median", round(TO.sw_next_below.median(), 1), "uV, negative in", int((TO.sw_next_below < 0).sum()),
      "of", int(TO.sw_next_below.notna().sum()))
print("channels with a positive sharp wave on the centre's shank, with larger rel_y than the centre:", int(TO.n_pos_above.sum()), "; with smaller:", int(TO.n_pos_below.sum()),
      "; sessions with any positive channel above / below:", int((TO.n_pos_above > 0).sum()), "/", int((TO.n_pos_below > 0).sum()))
rs = TO[TO.session.isin(RULE_SESS)]
print("sessions following the rule:", len(RULE_SESS), "of", len(TO), "; their theta channel lies above the centre (rel_y larger) in", int((rs.dy_theta > 0).sum()),
      "and has a positive sharp wave in", int((rs.sw_theta > 0).sum()))
print("sessions whose sharp wave uses all ripples (20 or fewer in homecage non-REM):", TO[TO.all_ripples].session.tolist())
print("theta channel same as the layer centre:", int((TO.theta_ch == TO.pyr_ch).sum()), "sessions")
print("theta cycles in homecage wake per session: median", int(TO.n_cycles_hc_wake.median()), "range", TO.n_cycles_hc_wake.min(), "-", TO.n_cycles_hc_wake.max(),
      "; on the centre channel median", int(TO.n_cycles_hc_wake_centre.median()))
hcm = ~TC.subject.isin(h.HOMECAGE_EXCLUDED)
print("clock check, light-responsive cells of homecage mice locked in wake at p < 0.01: phases as given",
      int((TC[hcm].wake_p < 0.01).sum()), "of", int(hcm.sum()), "; with spikes against the phase 10.3 s later",
      int((TC[hcm].wake_p_shift < 0.01).sum()), "; median wake MRL", round(TC[hcm].wake_mrl.median(), 4), "against", round(TC[hcm].wake_mrl_shift.median(), 4))
print(TO[["session", "where", "all_ripples", "dy_theta", "sw_theta", "sw_pyr", "sw_next_above", "sw_next_below", "n_pos_above", "n_pos_below"]].round(1).to_string())

> Claim fig1f-rem-shift: Earlier-born light-responsive neurons more often shift their preferred theta phase in REM
> sleep (30.89% at E13.5 to 19.7% at E16.5; regression slope -3.42% per day).

In [ ]:
TCH = TC[hcm].copy()
def rem_shift(df, thr):
    b = df[(df.wake_p < thr) & (df.rem_p < thr)].copy()
    wd, rd = np.degrees(b.wake_phase), np.degrees(b.rem_phase)
    b["shift"] = ((wd >= 120) & (wd <= 300)) & ~((rd >= 120) & (rd <= 300))
    return b
def slope_test(b, claim, k):
    rng = np.random.default_rng(seed(claim) + k)
    x = np.array([h.BD_DAY[d] for d in h.BIRTHDATES])
    grp = [b[b.birthdate == d]["shift"].values.astype(float) for d in h.BIRTHDATES]
    fr = np.array([g.mean() * 100 for g in grp])
    sl = np.polyfit(x, fr, 1)[0]
    boots = [np.polyfit(x, [rng.choice(g, len(g)).mean() * 100 for g in grp], 1)[0] for _ in range(1000)]
    pool = np.concatenate(grp)
    null = [np.polyfit(x, [rng.choice(pool, len(g)).mean() * 100 for g in grp], 1)[0] for _ in range(1000)]
    return fr, sl, np.percentile(boots, [2.5, 97.5]), float(np.mean(np.array(null) <= sl))
for thr in (0.01, 0.05):
    b = rem_shift(TCH, thr)
    print(f"Rayleigh threshold {thr}: cells locked in both states per birthdate {b.groupby('birthdate').size().tolist()}")
    for k in (SEEDS_PLUS if thr == 0.01 else [0]):
        fr, sl, ci, p = slope_test(b, "fig1f-rem-shift", k)
        print(f"  seed +{k}: % REM shifting {np.round(fr, 2).tolist()}, slope {sl:.2f} %/day, 95% CI {np.round(ci, 2).tolist()}, one-tailed p = {p:.3f}")
print("paper's n locked in both states (legend): [123, 106, 143, 76]")
for lab, df in (("variant, sessions whose channel follows the rule (positive sharp wave above the centre)", TCH[TCH.session.isin(RULE_SESS)]),
                ("variant, phase from the pyramidal layer centre channel", TCc[~TCc.subject.isin(h.HOMECAGE_EXCLUDED)])):
    b = rem_shift(df, 0.01)
    fr, sl, ci, p = slope_test(b, "fig1f-rem-shift", 0)
    print(f"{lab}: n {b.groupby('birthdate').size().tolist()}, % REM shifting {np.round(fr, 2).tolist()}, slope {sl:.2f} %/day, 95% CI {np.round(ci, 2).tolist()}, one-tailed p = {p:.3f}")
b = rem_shift(TCH, 0.01)
print("within tolerance (0.01):", [abs(a - e) <= t for a, e, t in zip(slope_test(b, 'fig1f-rem-shift', 0)[0], [30.89, 24.5, 23.8, 19.7], [3.09, 2.45, 2.38, 1.97])])
print("by mouse, % shifting (0.01):", (b.groupby(["birthdate", "subject"])["shift"].mean() * 100).round(1).to_dict())

**Result: not reproduced, resting on the choice of theta channel.** On the rule's channel 90, 75, 76 and 60 cells are locked in both states (the legend's n are 123, 106, 143 and 76), and the fractions of REM-shifting cells are 27.78, 21.33, 21.05 and 23.33%, all four outside tolerance; the slope is -1.36% per day (95% interval -5.8 to 2.9) with one-tailed p 0.269 to 0.301 over the five seeds, so it is not significant. No reading gives a significant slope: the code's Rayleigh threshold of 0.05 gives -2.54 (p = 0.092), the sessions whose channel follows the rule only -1.41 (p = 0.259), and the layer centre channel -2.01 (p = 0.183).

> Claim fig2d-theta-depth: Intermediate-birthdate neurons have lower theta depth modulation in waking (medians
> 0.127, 0.099, 0.094, 0.103) but not in REM sleep.

In [ ]:
w = TCH[TCH.wake_p < 0.01].rename(columns={"wake_mrl": "wake_depth"})
r = TCH[TCH.rem_p < 0.01].rename(columns={"rem_mrl": "rem_depth"})
mw, pw = by_bd(w, "wake_depth", "fig2d-theta-depth")
mr, pr = by_bd(r, "rem_depth", "fig2d-theta-depth")
print("wake: E14.5 and E15.5 both below E13.5 and E16.5:", max(mw[1], mw[2]) < min(mw[0], mw[3]),
      "; within tolerance:", [abs(a - e) <= 0.1 * e for a, e in zip(mw, [0.127, 0.099, 0.094, 0.103])])
print("by mouse (median wake depth):", w.groupby(["birthdate", "subject"]).wake_depth.median().round(3).to_dict())
for lab, df in (("variant, sessions whose channel follows the rule", TCH[TCH.session.isin(RULE_SESS)]),
                ("variant, pyramidal layer centre channel", TCc[~TCc.subject.isin(h.HOMECAGE_EXCLUDED)])):
    ww = df[df.wake_p < 0.01]
    g = [ww[ww.birthdate == b_].wake_mrl.values for b_ in h.BIRTHDATES]
    st, pv = h.boot_kw(g, rng=np.random.default_rng(seed("fig2d-theta-depth")))
    md = [float(np.median(x)) for x in g]
    print(f"{lab}: wake n {[len(x) for x in g]}, medians {np.round(md, 4).tolist()}, H = {st:.2f}, p = {pv:.4f}; "
          f"E14.5 and E15.5 below both: {max(md[1], md[2]) < min(md[0], md[3])}; within tolerance: {[abs(a - e) <= 0.1 * e for a, e in zip(md, [0.127, 0.099, 0.094, 0.103])]}")

**Result: reproduced, resting on the choice of theta channel.** Wake depth modulation medians 0.121, 0.090, 0.093 and 0.104 are within tolerance, E14.5 and E15.5 are below E13.5 and E16.5, and the bootstrap Kruskal-Wallis gives p < 0.0002 under all seeds; in REM sleep the medians are 0.141 to 0.148 and p = 0.64 to 0.66, so no difference is found, as the paper reports. The variants keep the profile and the test but each puts one median just outside tolerance: E14.5 0.089 on the sessions whose channel follows the rule, E16.5 0.115 on the layer centre channel.

> Claim fig2c-theta-cofiring: SBD pairs cofire more than DBD pairs in theta cycles (median 0.0189 against 0.0087).

In [ ]:
TPH = TP[~TP.subject.isin(h.HOMECAGE_EXCLUDED)]
s, d, p = sbd_dbd(TPH, "theta_corr")
print("pairs SBD", int(TPH.sbd.sum()), "DBD", int((~TPH.sbd).sum()), "; within tolerance:", abs(s - 0.0189) <= 0.0019, abs(d - 0.0087) <= 0.0009)
print("variant, sessions whose channel follows the rule:")
s2, d2, p2 = sbd_dbd(TPH[TPH.session.isin(RULE_SESS)], "theta_corr")
print("   within tolerance:", abs(s2 - 0.0189) <= 0.0019, abs(d2 - 0.0087) <= 0.0009)
print("variant, pyramidal layer centre channel:")
s3, d3, p3 = sbd_dbd(TPc[~TPc.subject.isin(h.HOMECAGE_EXCLUDED)], "theta_corr")
print("   within tolerance:", abs(s3 - 0.0189) <= 0.0019, abs(d3 - 0.0087) <= 0.0009)
print("per birthdate SBD/DBD medians:", TPH.groupby(["birthdate", "sbd"]).theta_corr.median().round(4).to_dict())

**Result: partially reproduced, resting on the choice of theta channel.** SBD median 0.0170 against DBD 0.0070 over 3,727 and 50,392 pairs with a correlation, rank-sum p = 1.3e-42; both medians lie just below their tolerance (0.0170 to 0.0208 and 0.0078 to 0.0096; the SBD median rounds to the lower bound but the printed check puts it below). On the sessions whose channel follows the rule (72 of the 104) the medians are 0.0164 and 0.0069; with the phase from the layer centre channel they are 0.0178 and 0.0079, both within tolerance.

> Claim fig6c-theta-rate-preserved: Individual pyramidal neurons preserve their theta-related firing rates
> between the familiar and novel environments (Spearman rho = 0.7378, n = 2,544 neurons).

In [ ]:
NS = S.loc[S.subject.isin(h.NOVEL_MICE), "session"].tolist()
TRN = TR[TR.session.isin(NS)].copy()
def theta_rates(wake_only, src=None):
    x = (TRN if src is None else src)
    x = x[x.wake_only == wake_only]
    fam = x[x.epoch == 0].set_index(["session", "unit"])
    nov = x[x.epoch > 0].groupby(["session", "unit"])[["spikes", "dur"]].sum()
    df = fam[["rate", "lr"]].rename(columns={"rate": "fam"}).join(nov)
    df["nov"] = df.spikes / df.dur
    return df.dropna(subset=["fam", "nov"])
print("novel sessions:", len(NS), "; novel subsessions:", TRN[TRN.epoch > 0].groupby(["session", "epoch"]).ngroups)
for wo in (True, False):
    df = theta_rates(wo)
    rho, p = stats.spearmanr(df.fam, df.nov)
    print(f"{'Methods, cycles in wake' if wo else 'code variant, no wake restriction'}: n = {len(df)} cells, Spearman rho = {rho:.4f}, p = {p:.3g}")
df = theta_rates(True)
print("per mouse rho:", {s: round(stats.spearmanr(g.fam, g.nov)[0], 3) for s, g in df.reset_index().assign(subject=lambda d: d.session.str[:8]).groupby("subject")})
print("theta cycles per subsession (wake only): median", int(TRN[TRN.wake_only].groupby(['session','epoch']).n_cycles.first().median()),
      "min", int(TRN[TRN.wake_only].groupby(['session','epoch']).n_cycles.first().min()))
dfc = theta_rates(True, TRc[TRc.session.isin(NS)])
print(f"variant, pyramidal layer centre channel: n = {len(dfc)}, Spearman rho = {stats.spearmanr(dfc.fam, dfc.nov)[0]:.4f}")
print("novel sessions whose channel follows the rule:", len(RULE_SESS & set(NS)), "of", len(NS))
dfr = theta_rates(True, TRN[TRN.session.isin(RULE_SESS)])
print(f"variant, the novel sessions whose channel follows the rule: n = {len(dfr)} cells, Spearman rho = {stats.spearmanr(dfr.fam, dfr.nov)[0]:.4f}, p = {stats.spearmanr(dfr.fam, dfr.nov)[1]:.3g}")


**Result: reproduced, resting on the choice of theta channel.** Over the 2,544 pyramidal cells of the 12 novel sessions (the paper's n), theta-related rates in the familiar and novel subsessions correlate with Spearman rho = 0.735, p < 1e-300; the same with or without the wake restriction, 0.723 to 0.742 in each mouse, 0.738 on the layer centre channel, and 0.734 over the 1,259 cells of the 5 novel sessions whose channel follows the rule.

> Claim fig6d-rate-lower-novel: Theta-related firing rates are slightly lower in novel environments than in the
> familiar one (median 0.69 against 0.77 Hz).

In [ ]:
for wo in (True, False):
    df = theta_rates(wo)
    p = stats.ranksums(df.nov, df.fam).pvalue
    pw = stats.wilcoxon(df.nov, df.fam).pvalue
    print(f"{'Methods (wake)' if wo else 'code variant'}: familiar median {df.fam.median():.3f} Hz, novel median {df.nov.median():.3f} Hz, "
          f"n = {len(df)}; rank-sum p = {p:.3g}; paired signed-rank p = {pw:.3g}")
dfc = theta_rates(True, TRc[TRc.session.isin(NS)])
print(f"variant, pyramidal layer centre channel: familiar median {dfc.fam.median():.3f} Hz, novel {dfc.nov.median():.3f} Hz, rank-sum p = {stats.ranksums(dfc.nov, dfc.fam).pvalue:.3g}")
dfr = theta_rates(True, TRN[TRN.session.isin(RULE_SESS)])
print(f"variant, the novel sessions whose channel follows the rule: n = {len(dfr)}, familiar median {dfr.fam.median():.3f} Hz, novel {dfr.nov.median():.3f} Hz, "
      f"rank-sum p = {stats.ranksums(dfr.nov, dfr.fam).pvalue:.3g}; paired signed-rank p = {stats.wilcoxon(dfr.nov, dfr.fam).pvalue:.3g}")


**Result: reproduced, resting on the choice of theta channel.** Median theta-related rate 0.757 Hz in the familiar maze and 0.672 Hz in novel environments over 2,544 cells, rank-sum p = 0.00093 (paired signed-rank p = 2.8e-7); both within tolerance. On the layer centre channel 0.755 and 0.676 Hz, p = 0.0026. On the 5 novel sessions whose channel follows the rule (1,259 cells) the medians are 0.704 and 0.643 Hz with rank-sum p = 0.051 and paired signed-rank p = 0.0047, so the rank-sum test the claim names falls just short of 0.05 on that subset.

## Familiar figure-eight maze

> Claim maze-cohort: Nine mice (E13.5 n = 2, E14.5 n = 2, E15.5 n = 2, E16.5 n = 3) ran a spatial alternation
> task on a familiar figure-eight maze, and three mice explored novel environments.

In [ ]:
fam = S[S.subject.isin(h.FAMILIAR_MICE) & (S.beh_epochs > 0)]
print("familiar-maze mice with figure-eight sessions:", fam.subject.nunique(), fam.groupby("birthdate").subject.nunique().to_dict(),
      "; sessions:", len(fam))
print("first environment of their sessions:", fam.envs.str.split(",").str[0].value_counts().to_dict())
nov = S[S.subject.isin(h.NOVEL_MICE)]
nov_has = nov[nov.envs.str.split(",").str.len() > 1]
print("novel-environment mice with a later subsession:", nov_has.subject.nunique(), sorted(nov_has.subject.unique()),
      "; sessions:", len(nov_has), "of", len(nov))
other = S[~S.subject.isin(h.FAMILIAR_MICE | h.NOVEL_MICE) & (S.beh_epochs > 0)]
print("other mice with behavioural subsessions:", other.subject.unique().tolist(), len(other), "sessions")

**Result: reproduced.** The nine mice of the table's Figure 5 column have 57 sessions with a first subsession on the figure-eight (2, 2, 2 and 3 mice per birthdate), and the three mice of the Figure 6 column have later subsessions in all 12 of their sessions.

In [ ]:
FM = [META[s] for s in fam.session]
t = time.time()
mz = h.run_sessions(h.maze_session, FM)
print(f"{time.time()-t:.0f} s, {len(mz)} sessions, pyramidal cells {sum(len(x['pyr']) for x in mz)}, light-responsive {sum(int(x['lr'].sum()) for x in mz)}")

> Claim maze-task: On the figure-eight maze each trial was 205 cm long, position was tracked at 30 Hz, and
> animals ran at least ten trials along each arm in every session.

In [ ]:
tl = pd.DataFrame([{"session": x["session"], "x0": t["x0"], "x1": t["x1"], "arm": t["arm"]} for x in mz for t in x["trials"]])
tl["extent"] = tl.x1 - tl.x0
print("linearized extent per trial (cm): median", round(tl.extent.median(), 1), "range", round(tl.extent.min(), 1), "-", round(tl.extent.max(), 1),
      "; max position", round(tl.x1.max(), 1))
rates = np.array([1 / x["dt"] for x in mz])
print("tracking rate (1/median interval): median", round(float(np.median(rates)), 2), "Hz, range", round(rates.min(), 2), "-", round(rates.max(), 2))
tc = pd.DataFrame([{"session": x["session"], **x["trial_counts"]} for x in mz])
tc["min_type"] = tc[["left", "right"]].min(axis=1); tc["total"] = tc.left + tc.right
print("sessions:", len(tc), "; with >= 10 of each type:", int((tc.min_type >= 10).sum()), "; with >= 20 trials:", int((tc.total >= 20).sum()))
print(tc[tc.min_type < 10].to_string())
# trials table itself, before any mapping filter: trials inside the familiar epoch by visited arm
tt = []
for x in mz:
    m = META[x["session"]]; t0, t1 = h.behaviour_epochs(m)[0][:2]; tr = m["trials"]
    arms = [a for s_, e_, a in zip(tr["start_time"], tr["stop_time"], tr["visited_arm"]) if s_ >= t0 - 1 and e_ <= t1 + 1]
    tt.append({"session": x["session"], "table_left": arms.count("left"), "table_right": arms.count("right"),
               "table_other": len(arms) - arms.count("left") - arms.count("right"), "table_all_rows": len(tr["start_time"]),
               "mapped_left": x["trial_counts"]["left"], "mapped_right": x["trial_counts"]["right"]})
TT = pd.DataFrame(tt); TT["table_min"] = TT[["table_left", "table_right"]].min(axis=1)
print("trials table: sessions with >= 10 of each arm", int((TT.table_min >= 10).sum()), "of", len(TT), "; with >= 20 left plus right",
      int((TT.table_left + TT.table_right >= 20).sum()), "; sessions where the mapped counts differ from the table:",
      int(((TT.table_left != TT.mapped_left) | (TT.table_right != TT.mapped_right)).sum()))
print("other visited_arm values:", collections.Counter(a for x in mz for a in META[x["session"]]["trials"]["visited_arm"] if a not in ("left", "right")))
print(TT.to_string())

**Result: not reproduced.** Trials span a median 202.8 cm of linearized position (maximum 205.0) and tracking runs at 30.0 Hz, both within tolerance, but only 30 of 57 sessions hold at least ten trials of each type, and 6 sessions hold fewer than 20 trials in all, so neither the full nor the partial condition on trial counts holds. The trials table itself, before any mapping filter, gives the same: 30 of 57 sessions with ten of each arm and 51 with at least 20; the counted trials differ from the table by one trial in two sessions, and no visited_arm value other than left or right occurs.

> Claim novel-explorations: The novel-environment mice made 16 novel environment explorations in six distinct
> environments (two linear tracks, three open fields and one figure-eight maze).

In [ ]:
ex = [(m["session_id"], e[2]) for m in metas if m["subject_id"] in h.NOVEL_MICE for e in h.behaviour_epochs(m)[1:]]
print("novel explorations (subsessions after the first):", len(ex))
print("by environment:", collections.Counter(e for _, e in ex))
print("distinct environments:", len(set(e for _, e in ex)))

**Result: reproduced.** The three novel mice have 16 subsessions after the familiar one, in six environments: two linear tracks, three open fields and one novel figure-eight (tmaze_ipshita).

> Claim fig5b-spatial-info-by-birthdate: Spatial information (medians 0.836, 1.05, 1.35, 0.88 bits/s) and spatial
> coverage (36, 44, 44, 18 cm) on the familiar maze are largest in intermediate-birthdate neurons.

In [ ]:
def cell_maze_measures(x):
    out = []
    M, occ = x["maps"], x["occ"]
    for c in range(M.shape[1]):
        rl, rr = M[0, c], M[1, c]
        cat, ocat = np.r_[rl, rr], np.r_[occ[0], occ[1]]
        def cov(rate, o, use_info):
            ok = np.isfinite(rate) & (o > 0)
            p = o[ok] / o[ok].sum(); r = rate[ok]; mr = np.sum(p * r)
            hi = r > 5
            if use_info:
                with np.errstate(divide="ignore", invalid="ignore"):
                    term = (r / mr) * np.log2(r / mr)
                hi = hi & (term > 1)
            return int(hi.sum())
        out.append({"session": x["session"], "subject": x["subject"], "birthdate": x["birthdate"], "lr": bool(x["lr"][c]), "unit": int(x["unit"][c]),
                    "si_cat": h.skaggs_info(cat, ocat),
                    "si_mean": np.nanmean([h.skaggs_info(rl, occ[0]), h.skaggs_info(rr, occ[1])]),
                    "cov_a_cat": cov(cat, ocat, True), "cov_b_cat": cov(cat, ocat, False),
                    "cov_a_mean": np.mean([cov(rl, occ[0], True), cov(rr, occ[1], True)]),
                    "cov_b_mean": np.mean([cov(rl, occ[0], False), cov(rr, occ[1], False)])})
    return out
MC = pd.DataFrame([r for x in mz for r in cell_maze_measures(x)])
ML = MC[MC.lr]
exp_si, exp_cov = [0.836, 1.05, 1.35, 0.88], [36, 44, 44, 18]
for col in ["si_cat", "si_mean"]:
    med, ps = by_bd(ML, col, "fig5b-spatial-info-by-birthdate", n_seeds=5 if col == "si_cat" else 1)
    print(f"  {col}: inverted-U {h.inverted_u(med)}, within tolerance {[abs(a - e) <= 0.1 * e for a, e in zip(med, exp_si)]}")
for col in ["cov_a_cat", "cov_b_cat", "cov_a_mean", "cov_b_mean"]:
    med, ps = by_bd(ML, col, "fig5b-spatial-info-by-birthdate", n_seeds=5 if col == "cov_a_cat" else 1)
    print(f"  {col}: inverted-U {h.inverted_u(med)}, within tolerance {[abs(a - e) <= 0.1 * e for a, e in zip(med, exp_cov)]}")
print("by mouse (median si_cat):", ML.groupby(["birthdate", "subject"]).si_cat.median().round(3).to_dict())

**Result: partially reproduced.** Both measures differ across birthdates (bootstrap Kruskal-Wallis p <= 0.0008 under all readings and seeds) and both have their highest median at E14.5 or E15.5 with E13.5 and E16.5 below it, as the claim states, but no reading brings all the medians within tolerance: spatial information is 0.56, 0.63, 0.92 and 0.62 bits/s (concatenated maps) against 0.836, 1.05, 1.35 and 0.88, that is 29.5 to 40% below, and coverage 29.5, 31.5, 31 and 12 cm (reading a) against 36, 44, 44 and 18. Under coverage reading (b) the E13.5 median (34.5 cm) is within tolerance and the other three are not.

> Claim fig5c-sbd-ratemap-overlap: SBD pairs have higher spatial ratemap correlations than DBD pairs on the
> familiar maze (median 0.0152 against -0.0256).

In [ ]:
STEM = slice(74, 205)  # bins 75 to 205, post_stem_inds (claims)
def corr_visited(a, b):
    ok = np.isfinite(a) & np.isfinite(b)
    if ok.sum() < 3 or a[ok].std() == 0 or b[ok].std() == 0:
        return np.nan
    return float(np.corrcoef(a[ok], b[ok])[0, 1])
def corr_code(a, b):
    if not (np.isfinite(a).all() and np.isfinite(b).all()) or a.std() == 0 or b.std() == 0:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])
rows = []
for x in mz:
    M = x["maps"]; V = np.concatenate([M[0][:, STEM], M[1][:, STEM]], axis=1)
    lr = np.where(x["lr"])[0]; nr = np.where(~x["lr"])[0]
    for sbd, pp in ((True, [(a, b) for i, a in enumerate(lr) for b in lr[i + 1:]]), (False, [(a, b) for a in lr for b in nr])):
        for a, b in pp:
            rows.append({"session": x["session"], "subject": x["subject"], "birthdate": x["birthdate"], "sbd": sbd, "a": a, "b": b,
                         "r": corr_visited(V[a], V[b]), "r_code": corr_code(V[a], V[b]),
                         "rL": corr_visited(M[0][a, STEM], M[0][b, STEM]), "rR": corr_visited(M[1][a, STEM], M[1][b, STEM]),
                         "dshank": abs(int(x["shank"][a]) - int(x["shank"][b])),
                         "dist": 250 * round(abs(float(x["x"][a]) - float(x["x"][b])) / 250),
                         "el_dist": float(np.hypot(x["x"][a] - x["x"][b], x["y"][a] - x["y"][b])),
                         "ua": int(x["unit"][a]), "ub": int(x["unit"][b])})
MP = pd.DataFrame(rows)
s, d, p = sbd_dbd(MP, "r")
print("within tolerance:", abs(s - 0.0152) <= 0.0015, abs(d + 0.0256) <= 0.0026)
s2, d2, p2 = sbd_dbd(MP, "r_code")
print("by mouse SBD/DBD medians:", MP.groupby(["subject", "sbd"]).r.median().round(4).to_dict())

**Result: partially reproduced.** Over 2,609 SBD and 39,965 DBD pairs (the paper's counts), SBD median 0.0189 against DBD -0.0239, rank-sum p = 5.9e-9; the DBD median is within tolerance and the SBD median is above it (0.0137 to 0.0167). The code's variant gives 0.0187 and -0.0249.

> Claim fig5d-overlap-by-distance: Ratemap overlap of SBD pairs is larger than that of DBD pairs within 250
> micrometres (same and adjacent shanks) but not at 500 micrometres.

In [ ]:
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
print("shank distance from the electrodes' x positions against |shank difference| x 250 um: agreeing in", int((MP.dist == 250 * MP.dshank).sum()), "of", len(MP),
      "pairs (files with Front and Back faces have both faces of each shank at the same x)")
D5 = MP[MP.dshank <= 2].dropna(subset=["r"]).copy()
D5["dist"] = D5.dshank * 250; D5["grp"] = np.where(D5.sbd, "SBD", "DBD")   # kcoords difference (criterion)
print("pairs by shank distance (um) and group:", D5.groupby(["dist", "grp"]).size().to_dict(), "; dropped at 750 um:", int((MP.dshank > 2).sum()))
fit = smf.ols("r ~ C(grp, Sum) * C(dist, Sum)", data=D5).fit()
print(sm.stats.anova_lm(fit, typ=3).round(4).to_string())
means = D5.groupby(["dist", "grp"]).r.mean()
print("means:", means.round(4).to_dict())
expd = {(0, "SBD"): 0.0707, (0, "DBD"): 0.0284, (250, "SBD"): 0.0445, (250, "DBD"): 0.0173, (500, "SBD"): 0.0195, (500, "DBD"): 0.0151}
print("within tolerance:", {k: abs(means[k] - v) <= 0.1 * v for k, v in expd.items()})
tk = pairwise_tukeyhsd(D5.r, D5.grp + "_" + D5.dist.astype(str))
tt = pd.DataFrame(tk.summary().data[1:], columns=tk.summary().data[0])
print(tt[[a.split("_")[1] == b.split("_")[1] for a, b in zip(tt.group1, tt.group2)]].to_string())
# The model above has the interaction, so its six cell means are the group means and its residual mean square is the
# six-group one: Tukey-Kramer over the six groups is the comparison MATLAB's multcompare makes on that anovan model.
print("residual mean square, two-way model", round(fit.mse_resid, 6), "; six-group one-way", round(smf.ols("r ~ C(grp):C(dist)", data=D5).fit().mse_resid, 6))

**Result: partially reproduced.** The birthdate main effect has p = 0.0001 over 2,561 SBD and 38,497 DBD pairs, and SBD exceeds DBD by Tukey's test at 0 um (p = 0.0002) and 250 um (p = 0.041) but not at 500 um (p = 0.96), as the paper says; five of the six means are outside the tight tolerances (for example SBD 0 um 0.060 against 0.0707).

> Claim fig5e-pv-decorrelation: Population firing-rate vectors at the same linearized position decorrelate between
> left and right trials once the animal leaves the common stem (median correlations stem 0.93, turn 1 0.412,
> arm 0.372, turn 2 0.394).

In [ ]:
SEG = {"stem": (0, 74), "turn 1": (74, 111), "arm": (111, 185), "turn 2": (185, 205)}
pv = []
for x in mz:
    M = x["maps"]
    for b in range(205):
        a1, a2 = M[0][:, b], M[1][:, b]
        ok = np.isfinite(a1) & np.isfinite(a2)
        r = np.corrcoef(a1[ok], a2[ok])[0, 1] if ok.sum() > 2 and a1[ok].std() > 0 and a2[ok].std() > 0 else np.nan
        seg = [k for k, (lo, hi) in SEG.items() if lo <= b < hi][0]
        pv.append({"session": x["session"], "bin": b, "seg": seg, "r": r})
PV = pd.DataFrame(pv)
print("PV correlations per segment:", PV.groupby("seg").r.agg(["median", "count"]).round(4).loc[list(SEG)].to_dict(), "; NaN:", int(PV.r.isna().sum()))
groups = [PV[PV.seg == k].r.dropna().values for k in SEG]
for k in SEEDS_PLUS:
    H_, p = h.boot_kw(groups, rng=np.random.default_rng(seed("fig5e-pv-decorrelation") + k), n_boot=5000 if k == 0 else 1000)
    print(f"seed +{k}: bootstrap Kruskal-Wallis H = {H_:.1f}, p = {p:.4f}")
meds = [np.median(g) for g in groups]
print("within tolerance:", [abs(a - e) <= 0.1 * e for a, e in zip(meds, [0.93, 0.412, 0.372, 0.3944])], "; stem highest:", int(np.argmax(meds)) == 0)

**Result: partially reproduced.** Median PV correlations are 0.930 on the stem, 0.488 on turn 1, 0.407 on the arm and 0.407 on turn 2; the stem is highest and the bootstrap Kruskal-Wallis gives p < 0.0002 under all seeds, but the turn 1 median is outside tolerance (0.371 to 0.453).

> Claim fig5f-remap-together: Between left and right trials, ratemaps of SBD pairs reorganize closer together than
> those of DBD pairs.

In [ ]:
LRp = MP.dropna(subset=["rL", "rR"])
obs = pd.concat([LRp.assign(x=LRp.rL, y=LRp.rR), LRp.assign(x=LRp.rR, y=LRp.rL)])
sb = obs[obs.sbd]
edges = np.percentile(np.r_[sb.rL, sb.rR], [0, 33, 66, 99])
print("tercile edges (SBD, both trial types):", np.round(edges, 4).tolist())
for lab, open_ in (("edges as the code has them", False), ("variant, outer edges open", True)):
    e = edges.copy()
    if open_: e[0], e[-1] = -np.inf, np.inf
    o = obs.copy(); o["ter"] = pd.cut(o.x, e, labels=[1, 2, 3], include_lowest=True)
    o = o.dropna(subset=["ter"]); o["grp"] = np.where(o.sbd, "SBD", "DBD")
    fit = smf.ols("y ~ C(grp, Sum) * C(ter, Sum)", data=o).fit()
    an = sm.stats.anova_lm(fit, typ=3)
    mt = o.groupby(["ter", "grp"], observed=True).y.mean().unstack()
    print(f"{lab}: n SBD {int(o.sbd.sum())}, DBD {int((~o.sbd).sum())}; birthdate F = {an.loc['C(grp, Sum)', 'F']:.2f}, p = {an.loc['C(grp, Sum)', 'PR(>F)']:.3g}; "
          f"tercile p = {an.loc['C(ter, Sum)', 'PR(>F)']:.3g}; interaction p = {an.loc['C(grp, Sum):C(ter, Sum)', 'PR(>F)']:.3g}")
    print("   tercile means:", mt.round(4).to_dict(), "; SBD > DBD in each tercile:", bool((mt.SBD > mt.DBD).all()))

**Result: reproduced.** The birthdate main effect has F = 24.0, p = 9.5e-7 with SBD higher, and SBD exceeds DBD in each tercile (0.020, 0.049, 0.072 against 0.002, 0.022, 0.043); the variant with open outer edges gives the same.

> Claim ed3-depth-control: Birthdate differences in single-cell features are not explained by anatomical depth
> alone; after matching depth distributions the bell-shaped birthdate profiles remain in most resamplings.

Depth of a neuron is the position of its maximum-waveform channel relative to the SPW reversal of its shank (criterion), the
reversal taken from the sharp-wave profile of the theta section (`pick_theta_channel`: the steepest positive-to-negative
step between neighbouring channels, interpolated to zero). On a shank without a reversal the shank's ripple-power centre
stands in (assumption; counted below, and the centre is used for every shank as a variant). Each resampling draws, per
birthdate, as many cells as the group holds, with replacement, so that the group's depth distribution in 20 um bins is
that of all light-responsive cells (`h.depth_match_draw`), and keeps the draw when the bootstrap Kruskal-Wallis test on depth
gives 0.4 < p < 0.6 (resample_depth.m, claims); the six measures are then tested with the bootstrap Kruskal-Wallis test.
The homecage measures are resampled over the 664 homecage light-responsive cells, the maze measures over the 466 maze ones.
To fit the run's time the inner bootstrap tests use 200 resamplings for the depth test and 300 for each measure (p
resolution 0.005 and 0.003, against thresholds of 0.4, 0.6 and 0.05); a draw is retried at most 100 times, and the four
extra seeds run 20 resamplings each.

In [ ]:
REV = {r["session"]: (r["rev"], r["cen"]) for _, r in TO.iterrows()}
def depth_of(df, centre_only=False):
    out, fb = [], 0
    for s_, sh_, y_ in zip(df.session, df.col, df.y):
        rev, cen = REV[s_]
        r = rev.get(int(sh_), np.nan)
        if centre_only or not np.isfinite(r):
            fb += not centre_only; r = cen.get(int(sh_), np.nan)
        out.append(y_ - r)
    return np.array(out), fb
HCD = LR.copy()
HCD["depth"], nfb = depth_of(HCD)
HCD["depth_centre"], _ = depth_of(HCD, centre_only=True)
th_ = TC[hcm & (TC.wake_p < 0.01)].set_index(["session", "unit"]).wake_mrl
HCD["theta_depth"] = [th_.get((s_, u_), np.nan) for s_, u_ in zip(HCD.session, HCD.unit)]
MZD = ML.merge(CELLS[["session", "unit", "col", "y"]], on=["session", "unit"], how="left")
MZD["depth"], nfb_m = depth_of(MZD)
MZD["depth_centre"], _ = depth_of(MZD, centre_only=True)
print("homecage light-responsive cells", len(HCD), "; shanks without a reversal, ripple centre used, cells:", nfb, "; maze cells", len(MZD), "fallback", nfb_m)
print("depth (um, positive above the reversal) by birthdate, median:", HCD.groupby("birthdate").depth.median().round(1).to_dict(),
      "; range", round(HCD.depth.min(), 1), "to", round(HCD.depth.max(), 1))
g0 = [HCD[HCD.birthdate == b].depth.values for b in h.BIRTHDATES]
print("depth across birthdates before matching: H = %.3f, p = %.4f (paper H = 26.327, p = 0)" % h.boot_kw_fast(g0, 5000, np.random.default_rng(seed("ed3-depth-control"))))
REF = HCD.depth.values
def ed3_run(df, cols, rng, n_res=500, n_boot=300, dcol="depth"):
    d = df[dcol].values; grp = df.birthdate.values; tries, hits, kept_p = 0, {c: 0 for c in cols}, []
    ok = np.isfinite(d)
    df, d, grp = df[ok], d[ok], grp[ok]
    for r_ in range(n_res):
        for k_ in range(100):
            idx = h.depth_match_draw(d, grp, REF, rng); tries += 1
            _, p_ = h.boot_kw_fast([d[idx][grp[idx] == b] for b in h.BIRTHDATES], 200, rng)
            if 0.4 < p_ < 0.6: break
        kept_p.append(p_)
        sub = df.iloc[idx]
        for c in cols:
            _, pc = h.boot_kw_fast([sub[sub.birthdate == b][c].dropna().values for b in h.BIRTHDATES], n_boot, rng)
            hits[c] += pc < 0.05
    return {c: hits[c] / n_res for c in cols}, tries / n_res, float(np.mean([0.4 < p < 0.6 for p in kept_p]))
t = time.time()
E3 = {}
for k in SEEDS_PLUS:
    n_res = 500 if k == 0 else 20
    fh, tr_h, okh = ed3_run(HCD, ["burst", "rip_rate", "rip_part", "theta_depth"], np.random.default_rng(seed("ed3-depth-control") + k), n_res)
    fm, tr_m, okm = ed3_run(MZD, ["si_cat", "cov_a_cat"], np.random.default_rng(seed("ed3-depth-control") + 1 + k), n_res)
    E3[k] = {**fh, **fm}
    print(f"seed +{k} ({n_res} resamplings): fractions with p < 0.05 {dict((c, round(v, 3)) for c, v in E3[k].items())}; draws per kept resampling {tr_h:.1f} and {tr_m:.1f}; "
          f"kept draws with 0.4 < depth p < 0.6: {okh:.2f} and {okm:.2f}")
exp3 = {"burst": 0.712, "rip_rate": 0.988, "rip_part": 0.938, "theta_depth": 1.0, "si_cat": 0.726, "cov_a_cat": 0.738}
print("within tolerance (claim's seed):", {c: abs(E3[0][c] - v) <= 0.1 * v for c, v in exp3.items()})
fc, _, _ = ed3_run(HCD, ["burst", "rip_rate", "rip_part", "theta_depth"], np.random.default_rng(seed("ed3-depth-control") + 2), 50, dcol="depth_centre")
print("variant, depth from the ripple-power centre of every shank (50 resamplings):", {c: round(v, 3) for c, v in fc.items()})
print(f"{time.time()-t:.0f} s")

**Result: not reproduced, resting on the depth reference.** Depth differs across birthdates before matching (bootstrap H = 23.5, p < 0.0002; the paper's H = 26.3), and every kept resampling has 0.4 < p < 0.6 on depth (5.8 and 5.0 draws per kept resampling), so the matching holds. The fractions of matched resamplings with p < 0.05 are 0.980 (burst index), 0.990 (SPW-R rate), 0.992 (SPW-R participation), 1.000 (wake theta depth), 0.980 (spatial information) and 0.992 (coverage). SPW-R rate, participation and theta depth are within tolerance; burst index, spatial information and coverage lie above the paper's 0.712, 0.726 and 0.738 beyond tolerance. The criterion's partial outcome needs the burst fraction within tolerance, so the outcome is not reproduced, although the birthdate differences survive depth matching more often than in the paper. The four extra seeds (20 resamplings each) give 0.90 to 1.0 for every measure, and depth from the ripple-power centre on every shank gives 0.92 to 1.0 for the homecage measures, the burst fraction outside tolerance under each (0.90 to 1.0 over the extra seeds, 0.92 on the ripple-power centre). 376 of the 664 homecage cells lie on shank faces without a sharp-wave reversal, where the ripple-power centre stands in.

## Novel environments

Rate maps in the familiar figure-eight and in the later subsessions of the 12 sessions of the three novel mice
(`h.novel_session`). The novel figure-eight and the linear tracks use their linearized position, which the files give
in centimeters, in 1 cm bins (the table prints each series, its unit and bin count). The open fields have only the 2D
head position, in the lab's units with no scale; they are binned assuming one unit is 1 cm, and at 2, 0.5 and 0.85 cm per unit as
variants. The evidence for the assumption is printed below the subsession table: where a lab-unit 2D series that is
not normalized to 0-1 runs beside a linearized series in cm (two linear tracks and the novel figure-eight), the
simultaneous speeds give 0.85 to 0.93 cm per unit, against 1.00 to 1.19 for the familiar maze's 2D series, which the
files give in cm; the open fields themselves carry no such check. Maps are in 1 cm x 1 cm bins smoothed with a 4.5 cm FWHM Gaussian (Methods). The linear_manu
subsession of e16_3m2_211211 has no position series inside its epoch: the series named for it lies inside the
openField_thomas epoch (printed below), so it is left out.

In [ ]:
t = time.time()
nv = h.run_sessions(h.novel_session, [META[s] for s in NS])
NE = pd.DataFrame([{"session": x["session"], "epoch": e["epoch"], "env": e["env"], "used": e["maps"] is not None, "kind": e.get("kind", ""),
                    "series": e.get("series", ""), "unit": e.get("unit", ""), "bins": e.get("nbins"),
                    "why": e.get("why", ""), "trials": e.get("trial_counts")} for x in nv for e in x["epochs"]])
print(f"{time.time()-t:.0f} s"); print(NE.to_string())
print("novel subsessions with maps:", int(NE[(NE.epoch > 0) & NE.used].shape[0]), "of", int((NE.epoch > 0).sum()))
# the one novel subsession without maps: where its position series lies
m = META["e16_3m2_211211"]
print("e16_3m2_211211 epochs:", [(e[2], round(e[0]), round(e[1])) for e in h.behaviour_epochs(m)])
print("e16_3m2_211211 position series:", {k: (round(v["t0"]), round(v["t1"]), v["unit"]) for k, v in m["series"].items()})

In [ ]:
# The scale of the 2D positions (assumption for the open fields; h.position_scale_check)
PSC = pd.DataFrame([r for rows in h.run_sessions(h.position_scale_check, [META[s] for s in NS]) for r in rows])
print(PSC.round(2).to_string())
nz = PSC[(PSC.unit != "centimeters") & PSC.cm_per_unit_from_speed.notna()]
print("cm per unit from the simultaneous speeds: familiar figure-eight (2D series in cm)",
      [round(float(v), 2) for v in PSC[PSC.unit == "centimeters"].cm_per_unit_from_speed.agg(["min", "median", "max"])],
      "; lab-unit series spanning more than 2 units", nz[nz.x_max - nz.x_min > 2].cm_per_unit_from_speed.round(2).tolist(),
      "; series spanning 0 to 1", nz[nz.x_max - nz.x_min <= 2].cm_per_unit_from_speed.round(1).tolist())
op = PSC[PSC.env.str.startswith("openField")]
fam2d = PSC[PSC.unit == "centimeters"].set_index("session").speed_p90
print("open fields: extent in lab units", {f"{r.session} {r.env}": (round(r.x_max - r.x_min, 2), round(r.y_max - r.y_min, 2)) for r in op.itertuples()})
print("open fields: 90th percentile of 1 s speed, lab units/s, against the same session's familiar 2D speed in cm/s:",
      {f"{r.session} {r.env}": (round(r.speed_p90, 1), round(float(fam2d[r.session]), 1)) for r in op.itertuples()})


In [ ]:
NOVEL_BINS = {"tmaze_ipshita": slice(50, 164)}  # novel figure-eight bins 51-164 (code, claims); linear tracks all bins
nrows, crows = [], []
for x in nv:
    lr = np.where(x["lr"])[0]; nr = np.where(~x["lr"])[0]
    pairs = [(True, a, b) for i, a in enumerate(lr) for b in lr[i + 1:]] + [(False, a, b) for a in lr for b in nr]
    for e in x["epochs"]:
        if e["maps"] is None: continue
        if e["kind"] == "open":   # 2D maps, flattened; the lab-unit scale as assumed (1 cm per unit) and two variants
            versions = [(s, M[0], occ[0]) for s, (M, occ) in e["scales"].items()]
        else:
            M, occ = e["maps"], e["occ"]
            sl = STEM if e["epoch"] == 0 else NOVEL_BINS.get(e["env"], slice(None))
            versions = [(1.0, np.concatenate([M[0][:, sl], M[1][:, sl]], axis=1), None)]
        for scale, V, occ1 in versions:
            for c in range(V.shape[0]):
                if e["kind"] == "open":
                    cat, ocat = V[c], occ1
                else:
                    cat, ocat = np.r_[M[0, c], M[1, c]], np.r_[occ[0], occ[1]]
                ok = np.isfinite(cat) & (ocat > 0)
                r = cat[ok]; mr = r.mean()
                with np.errstate(divide="ignore", invalid="ignore"):
                    eq = float(np.nanmean(np.where(r > 0, r * np.log2(r / mr), 0))) if mr > 0 else np.nan
                crows.append({"session": x["session"], "epoch": e["epoch"], "env": e["env"], "kind": e["kind"], "scale": scale, "unit": c,
                              "lr": bool(x["lr"][c]), "si": h.skaggs_info(cat, ocat), "si_equal": eq})
            for sbd, a, b in pairs:
                nrows.append({"session": x["session"], "epoch": e["epoch"], "env": e["env"], "kind": e["kind"], "scale": scale,
                              "sbd": sbd, "a": a, "b": b, "r": corr_visited(V[a], V[b])})
NP_ALL = pd.DataFrame(nrows); NC_ALL = pd.DataFrame(crows)
def at_scale(df, s): return df[(df.kind == "linear") | (df.scale == s)]
NP, NC = at_scale(NP_ALL, 1.0), at_scale(NC_ALL, 1.0)
print("cells x subsessions:", len(NC), "; pairs x subsessions:", len(NP), "; NaN correlations:", int(NP.r.isna().sum()))
print("open-field bins visited per subsession (1 cm per unit):", NC[NC.kind == "open"].groupby(["session", "env"]).size().to_dict())

> Claim fig6d-info-lower-novel: Spatial information is lower in novel environments than in the familiar one
> (median 0.39 against 0.88 bits/s).

In [ ]:
for col in ("si", "si_equal"):
    f_, n_ = NC[NC.epoch == 0][col].dropna(), NC[NC.epoch > 0][col].dropna()
    p = stats.ranksums(n_, f_).pvalue
    print(f"{col}: familiar median {f_.median():.3f} bits/s (n = {len(f_)}), novel median {n_.median():.3f} (n = {len(n_)}), rank-sum p = {p:.3g}")
print("novel medians by environment:", NC[NC.epoch > 0].groupby("env").si.median().round(3).to_dict())
for s_ in h.OPEN_SCALES[1:]:
    d_ = at_scale(NC_ALL, s_); f_, n_ = d_[d_.epoch == 0].si.dropna(), d_[d_.epoch > 0].si.dropna()
    print(f"variant, open fields at {s_} cm per unit: familiar {f_.median():.3f}, novel {n_.median():.3f} (n = {len(n_)}), rank-sum p = {stats.ranksums(n_, f_).pvalue:.3g}")
d_ = NC[NC.kind != "open"]; f_, n_ = d_[d_.epoch == 0].si.dropna(), d_[d_.epoch > 0].si.dropna()
print(f"variant, open fields left out: novel {n_.median():.3f} (n = {len(n_)}), rank-sum p = {stats.ranksums(n_, f_).pvalue:.3g}")

**Result: partially reproduced.** Spatial information is lower in novel environments than on the familiar maze (median 0.260 over 3,289 cell-environment values against 0.661 bits/s over 2,532 cells, rank-sum p = 5.0e-100), but both medians are outside tolerance. The same holds under every reading of the open-field scale (novel 0.305 at 2 cm per unit, 0.226 at 0.5, 0.251 at 0.85) and without the open fields (0.367, within tolerance, with the familiar median still outside), all with p < 1e-35.

> Claim fig6c-novel-ratemap-corr-higher: Pairwise spatial ratemap correlations are higher in novel environments
> (median 0.048) than in the familiar environment (median -0.045).

In [ ]:
f_, n_ = NP[NP.epoch == 0].r.dropna(), NP[NP.epoch > 0].r.dropna()
p = stats.ranksums(n_, f_).pvalue
print(f"familiar median {f_.median():.4f} (n = {len(f_)}), novel median {n_.median():.4f} (n = {len(n_)}), rank-sum p = {p:.3g}")
print("within tolerance:", abs(f_.median() + 0.045) <= 0.0045, abs(n_.median() - 0.048) <= 0.0048)
print("novel medians by environment:", NP[NP.epoch > 0].groupby("env").r.median().round(4).to_dict())
for s_ in h.OPEN_SCALES[1:]:
    d_ = at_scale(NP_ALL, s_); f_, n_ = d_[d_.epoch == 0].r.dropna(), d_[d_.epoch > 0].r.dropna()
    print(f"variant, open fields at {s_} cm per unit: familiar {f_.median():.4f}, novel {n_.median():.4f} (n = {len(n_)}), rank-sum p = {stats.ranksums(n_, f_).pvalue:.3g}")
d_ = NP[NP.kind != "open"]; f_, n_ = d_[d_.epoch == 0].r.dropna(), d_[d_.epoch > 0].r.dropna()
print(f"variant, open fields left out: novel {n_.median():.4f} (n = {len(n_)}), rank-sum p = {stats.ranksums(n_, f_).pvalue:.3g}")

**Result: not reproduced, resting on the assumed open-field scale.** With the open fields at 1 cm per lab unit, the novel median pairwise ratemap correlation is -0.031 over 39,702 pair-environment values (the paper has 41,991) against -0.045 on the familiar maze, rank-sum p = 0.78: higher, but not significantly. The outcome depends on that scale: at 0.85 cm per unit, the scale the simultaneous speeds give for the lab-unit series of the linear tracks and the novel figure-eight (0.85, 0.88 and 0.93 cm per unit, against 1.00 to 1.19 for the familiar maze's 2D series in cm), p = 0.57, and without the open fields p = 0.75, the same outcome; at 2 cm per unit p = 2.1e-6 and at 0.5 cm per unit p = 0.010, which would give partially reproduced. The data do not give the open fields' scale: their positions start at 0 and end at round values (31.5 x 35.0, 22.5 x 22.5 units) or near them (about 43 x 52), and their 90th-percentile speeds, 4.7 to 11.6 units/s, are at or below the same sessions' 11.2 to 15.8 cm/s on the familiar maze (in six of seven below), which fits a scale near 1 and does not rule out 2. The familiar median is within tolerance; the novel one (0.048 expected) is not under any reading (-0.025 to -0.033).

> Claim fig6e-sbd-novel-overlap: In novel environments SBD pairs have higher ratemap correlations than DBD pairs
> (median 0.117 against 0.046).

In [ ]:
s, d, p = sbd_dbd(NP[NP.epoch > 0], "r")
print("within tolerance:", abs(s - 0.117) <= 0.0117, abs(d - 0.046) <= 0.0046)
print("by environment SBD/DBD medians:", NP[NP.epoch > 0].groupby(["env", "sbd"]).r.median().round(4).to_dict())
for s_ in h.OPEN_SCALES[1:]:
    d_ = at_scale(NP_ALL, s_); d_ = d_[d_.epoch > 0]
    a_, b_ = d_[d_.sbd].r.dropna(), d_[~d_.sbd].r.dropna()
    print(f"variant, open fields at {s_} cm per unit: SBD {a_.median():.4f} DBD {b_.median():.4f}, rank-sum p = {stats.ranksums(a_, b_).pvalue:.3g}")
d_ = NP[(NP.epoch > 0) & (NP.kind != "open")]
print(f"variant, open fields left out: SBD {d_[d_.sbd].r.median():.4f} DBD {d_[~d_.sbd].r.median():.4f}, p = {stats.ranksums(d_[d_.sbd].r.dropna(), d_[~d_.sbd].r.dropna()).pvalue:.3g}")

**Result: partially reproduced.** In novel environments SBD pairs correlate more than DBD pairs (median 0.021 against -0.033 over 1,390 and 38,312 values, rank-sum p = 0.0011), but both medians are outside tolerance (0.117 and 0.046 expected). The same holds at 2, 0.5 and 0.85 cm per unit (p = 0.00023, 0.0037 and 0.0012) and without the open fields (p = 0.002).

> Claim fig6f-remap-together-novel: Ratemaps of SBD pairs remain closer together than those of DBD pairs between
> the familiar and novel environments.

In [ ]:
famr = NP[NP.epoch == 0][["session", "a", "b", "sbd", "r"]].rename(columns={"r": "fam"})
novr = NP[NP.epoch > 0][["session", "a", "b", "epoch", "r"]].rename(columns={"r": "nov"})
J = novr.merge(famr, on=["session", "a", "b"]).dropna(subset=["fam", "nov"])
J["grp"] = np.where(J.sbd, "SBD", "DBD")
ea = np.percentile(famr.fam.dropna(), [0, 33, 66, 99])  # each familiar pair once (legend)
eb = np.percentile(np.r_[famr.fam.dropna(), novr.nov.dropna()], [0, 33, 66, 99])
for lab, e in (("(a) familiar SBD and DBD correlations", ea), ("(b) code, all three columns", eb)):
    e = e.copy(); e[0], e[-1] = -np.inf, np.inf
    o = J.copy(); o["ter"] = pd.cut(o.fam, e, labels=[1, 2, 3])
    fit = smf.ols("nov ~ C(grp, Sum) * C(ter, Sum)", data=o).fit()
    an = sm.stats.anova_lm(fit, typ=3)
    mt = o.groupby(["ter", "grp"], observed=True).nov.mean().unstack()
    print(f"{lab}: n SBD {int(o.sbd.sum())}, DBD {int((~o.sbd).sum())}; birthdate F = {an.loc['C(grp, Sum)', 'F']:.2f}, p = {an.loc['C(grp, Sum)', 'PR(>F)']:.3g}; "
          f"tercile p = {an.loc['C(ter, Sum)', 'PR(>F)']:.3g}; interaction p = {an.loc['C(grp, Sum):C(ter, Sum)', 'PR(>F)']:.3g}")
    print("   tercile means:", mt.round(4).to_dict(), "; terciles with SBD > DBD:", int((mt.SBD > mt.DBD).sum()),
          "; SBD mean over all", round(o[o.sbd].nov.mean(), 4), "DBD", round(o[~o.sbd].nov.mean(), 4))
print("familiar pairs entering twice (sessions with two novel environments):", int(J.duplicated(["session", "a", "b"]).sum()))
# variant: one novel value per pair, the mean over its novel environments
J1 = J.groupby(["session", "a", "b", "sbd", "grp"], as_index=False).agg(nov=("nov", "mean"), fam=("fam", "first"))
e = ea.copy(); e[0], e[-1] = -np.inf, np.inf
o = J1.copy(); o["ter"] = pd.cut(o.fam, e, labels=[1, 2, 3])
an = sm.stats.anova_lm(smf.ols("nov ~ C(grp, Sum) * C(ter, Sum)", data=o).fit(), typ=3)
print(f"variant, one value per pair (reading a edges): n SBD {int(o.sbd.sum())}, DBD {int((~o.sbd).sum())}; birthdate F = {an.loc['C(grp, Sum)', 'F']:.2f}, p = {an.loc['C(grp, Sum)', 'PR(>F)']:.3g}")
print("Supplementary Table 5 tercile means, for comparison: SBD 0.0952, 0.1566, 0.15; DBD 0.0851, 0.0846, 0.0989")
def remap_novel_p(NPx):
    fr = NPx[NPx.epoch == 0][["session", "a", "b", "sbd", "r"]].rename(columns={"r": "fam"})
    nr = NPx[NPx.epoch > 0][["session", "a", "b", "epoch", "r"]].rename(columns={"r": "nov"})
    j = nr.merge(fr, on=["session", "a", "b"]).dropna(subset=["fam", "nov"]); j["grp"] = np.where(j.sbd, "SBD", "DBD")
    e = np.percentile(fr.fam.dropna(), [0, 33, 66, 99]); e[0], e[-1] = -np.inf, np.inf
    j["ter"] = pd.cut(j.fam, e, labels=[1, 2, 3])
    an = sm.stats.anova_lm(smf.ols("nov ~ C(grp, Sum) * C(ter, Sum)", data=j).fit(), typ=3)
    mt = j.groupby(["ter", "grp"], observed=True).nov.mean().unstack()
    return an.loc["C(grp, Sum)", "F"], an.loc["C(grp, Sum)", "PR(>F)"], int((mt.SBD > mt.DBD).sum()), j[j.sbd].nov.mean() > j[~j.sbd].nov.mean()
for s_ in h.OPEN_SCALES[1:]:
    F_, p_, k_, hi_ = remap_novel_p(at_scale(NP_ALL, s_))
    print(f"variant (a), open fields at {s_} cm per unit: birthdate F = {F_:.2f}, p = {p_:.3g}; SBD higher overall: {hi_}; terciles with SBD > DBD: {k_}")
F_, p_, k_, hi_ = remap_novel_p(NP[NP.kind != "open"])
print(f"variant (a), open fields left out: birthdate F = {F_:.2f}, p = {p_:.3g}; SBD higher overall: {hi_}; terciles with SBD > DBD: {k_}")


**Result: reproduced, resting on the assumed open-field scale.** Under both readings of the tercile edges the birthdate main effect has p = 0.0054 and 0.0038 with SBD higher (mean novel correlation 0.036 against 0.013 over 1,380 SBD and 37,762 DBD observations); SBD exceeds DBD in the middle and upper terciles, not the lowest, and the interaction has p = 0.0073 under reading (a). Counting each pair once in the ANOVA (the mean over its novel environments) gives p = 0.0081. Under reading (a) the main effect holds with SBD higher at 2, 0.5 and 0.85 cm per unit (p = 0.0018, 0.023 and 0.0060) and without the open fields (p = 0.016, SBD above DBD in all three terciles). The tercile means (SBD 0.004, 0.030, 0.072; DBD 0.008, 0.015, 0.015) are all below those of Supplementary Table 5 (SBD 0.0952, 0.1566, 0.15; DBD 0.0851, 0.0846, 0.0989), as the novel correlations as a whole are lower here (fig6c-novel-ratemap-corr-higher).

## Monosynaptic connections

Pyramidal-to-interneuron CCGs in 0.8 ms bins from spikes outside the light pulse periods, over the 96
homecage-mouse sessions (`h.mono_session`). The Methods give the test as identical to the reliability test
of the light-responsive detection (hollowed Gaussian baseline, 15 ms s.d., Poisson tests at alpha 0.001
Bonferroni corrected, the peak against the bins before zero lag) and refer to English et al. (2017) for the
rest. Assumed: the lags tested (0.8 to 2.8 ms, the window of the transmission measure), the hollow fraction (0.6,
the value of Stark and Abeles 2009 that English et al. use), and minimums of 100 presynaptic spikes and 50 CCG counts.
The second cell changes each in turn: the reliability test's literal "3 bins surrounding" the peak, hollow 0.4 and
0.8, kernel s.d. 7 and 10 ms, no minimums, and alpha 1e-5.

In [ ]:
t = time.time()
mo = h.run_sessions(h.mono_session, HM)
CON_ALL = pd.DataFrame([r for x in mo for r in x["rows"]])
CON = CON_ALL[CON_ALL.v_main].reset_index(drop=True)
npairs = sum(x["conn"].size for x in mo)
print(f"{time.time()-t:.0f} s; possible pyramidal-interneuron pairs {npairs}; tested (>= 100 presynaptic spikes, CCG with >= 50 counts) "
      f"{sum(x['n_tested'] for x in mo)}; connections {int(sum(x['conn'].sum() for x in mo))}")

In [ ]:
# The detector's assumed details, one at a time (h.MONO_VARIANTS): connections, fig7b convergence, the share of pyramidal
# pairs with a shared target (fig7fg), the transmission peak (fig7d) and the transmission medians of light-responsive presynaptic cells (fig7ce).
centres = np.sqrt(h.ISI_EDGES[:-1] * h.ISI_EDGES[1:]) * 1000
vrows = []
for v, spec in h.MONO_VARIANTS.items():
    sb_, db_, nz_, tot_ = [], [], 0, 0
    for x in mo:
        c = x["flags"][v]; lr = np.where(x["lr"])[0]; nr = np.where(~x["lr"])[0]
        sb_ += [h.convergence(c, a, b) for i, a in enumerate(lr) for b in lr[i + 1:]]
        db_ += [h.convergence(c, a, b) for a in lr for b in nr]
        ci = c.astype(int); sh = ci @ ci.T; iu = np.triu_indices(len(ci), 1); nz_ += int((sh[iu] > 0).sum()); tot_ += len(iu[0])
    sb_, db_ = np.array(sb_, float), np.array(db_, float)
    sb_, db_ = sb_[np.isfinite(sb_)], db_[np.isfinite(db_)]
    cv_ = CON_ALL[CON_ALL["v_" + v]]
    mean_ = np.nanmean(np.array(cv_.stp_isi.tolist(), float), 0)
    stp_ = cv_[cv_.pre_lr].groupby("birthdate").stp.median()
    vrows.append({"variant": v, **{k: str(val) for k, val in spec.items()}, "connections": int(sum(x["flags"][v].sum() for x in mo)),
                  "SBD_cv": np.median(sb_), "DBD_cv": np.median(db_), "p": stats.ranksums(sb_, db_).pvalue,
                  "pct_shared": 100 * nz_ / tot_, "peak_ms": centres[int(np.nanargmax(mean_))],
                  "stp_medians": np.round(stp_.values, 4).tolist()})
VR = pd.DataFrame(vrows)
print(VR.drop(columns=[c for c in VR.columns if c not in ("variant", "connections", "SBD_cv", "DBD_cv", "p", "pct_shared", "peak_ms", "stp_medians")]).round(4).to_string())
print("paper: 31,932 connections, convergence 0.2727 and 0.2, 65.27% shared, peak in 15-23 ms; settings:", {k: v for k, v in h.MONO_VARIANTS.items()})

> Claim fig7b-sbd-convergence: SBD pyramidal pairs converge onto the same interneurons more than DBD pairs (median
> convergence 0.2727 against 0.2).

In [ ]:
crow = []
for x in mo:
    c = x["conn"]; lr = np.where(x["lr"])[0]; nr = np.where(~x["lr"])[0]
    for i, a in enumerate(lr):
        for b in lr[i + 1:]: crow.append({"session": x["session"], "subject": x["subject"], "birthdate": x["birthdate"], "sbd": True, "cv": h.convergence(c, a, b)})
        for b in nr: crow.append({"session": x["session"], "subject": x["subject"], "birthdate": x["birthdate"], "sbd": False, "cv": h.convergence(c, a, b)})
CV = pd.DataFrame(crow)
s, d, p = sbd_dbd(CV, "cv")
print("within tolerance:", abs(s - 0.2727) <= 0.0273, abs(d - 0.2) <= 0.02)
print("by birthdate SBD/DBD medians:", CV.groupby(["birthdate", "sbd"]).cv.median().round(4).to_dict())

**Result: partially reproduced, resting on the assumed connection detector.** SBD pairs converge more than DBD pairs (median 0.455 against 0.364, rank-sum p = 1.3e-79), but both medians are far above the paper's (0.273 and 0.200); the detector finds 50,618 connections against the paper's 31,932. No variant of the detector's assumed details brings the count near the paper's (49,564 to 50,756 connections, 38,402 at alpha 1e-5), and under every variant SBD exceeds DBD, with SBD medians of 0.385 to 0.455 and DBD medians of 0.278 to 0.368, all above the paper's 0.273 and 0.200.

> Claim fig7ce-convergence-by-birthdate: Convergence onto interneurons (DBD-corrected medians 0.0857, 0.1071, 0.1,
> 0.0285) and spike transmission probability (0.0184, 0.0202, 0.0222, 0.0199) are bell-shaped functions of the
> presynaptic neurons' birthdate.

In [ ]:
sb = CV[CV.sbd].dropna().copy()
dm = CV[~CV.sbd].groupby("birthdate").cv.median()
sb["cv_corr"] = sb.cv - sb.birthdate.map(dm)
m1, p1 = by_bd(sb, "cv_corr", "fig7ce-convergence-by-birthdate")
print("corrected convergence: inverted-U", h.inverted_u(m1), "within tolerance", [abs(a - e) <= 0.1 * abs(e) for a, e in zip(m1, [0.0857, 0.1071, 0.1, 0.0285])])
L7 = CON[CON.pre_lr]
m2, p2 = by_bd(L7, "stp", "fig7ce-convergence-by-birthdate")
print("transmission: inverted-U", h.inverted_u(m2),
      "; within tolerance, legend:", [abs(a - e) <= 0.1 * e for a, e in zip(m2, [0.0184, 0.0202, 0.0222, 0.0199])],
      "; table:", [abs(a - e) <= 0.1 * e for a, e in zip(m2, [0.0238, 0.02356, 0.0247, 0.0195])])

**Result: partially reproduced, resting on the assumed connection detector.** Corrected convergence (medians 0.117, 0.143, 0.100, 0.054) and transmission probability (0.0141, 0.0153, 0.0172, 0.0150) are both inverted U across birthdates with bootstrap Kruskal-Wallis p < 0.0002, but only one median of the eight (corrected convergence, E15.5) is within tolerance under either reading of the transmission medians. Under the detector variants at most two of the four transmission medians are within tolerance of the legend's (alpha 1e-5).

> Claim fig7d-transmission-gamma: Spike transmission probability peaks at presynaptic interspike intervals of 15 to
> 23 ms, in the gamma range, with short-term depression at shorter intervals.

In [ ]:
isi = np.array(CON.stp_isi.tolist(), float)
centres = np.sqrt(h.ISI_EDGES[:-1] * h.ISI_EDGES[1:]) * 1000
mean = np.nanmean(isi, 0)
print(pd.DataFrame({"isi_lo_ms": h.ISI_EDGES[:-1] * 1000, "isi_hi_ms": h.ISI_EDGES[1:] * 1000, "centre_ms": centres,
                    "mean_stp": mean, "n": np.isfinite(isi).sum(0)}).round(4).to_string())
k = int(np.nanargmax(mean))
print(f"peak bin {h.ISI_EDGES[k]*1000:.1f}-{h.ISI_EDGES[k+1]*1000:.1f} ms (centre {centres[k]:.1f}); in 13.5-25.3 ms: {13.5 <= centres[k] <= 25.3}; "
      f"shortest bin {mean[0]:.4f} below peak {mean[k]:.4f}: {mean[0] < mean[k]}")
for lab, sel in (("light-responsive presynaptic only", CON.pre_lr.values),):
    mm = np.nanmean(isi[sel], 0); kk = int(np.nanargmax(mm)); print(lab, "peak centre", round(centres[kk], 1), "ms")
# how firm is the peak bin: connections resampled with replacement, the claim's seed and four more
inside = (centres >= 13.5) & (centres <= 25.3)
for k_s in SEEDS_PLUS:
    rng7d = np.random.default_rng(seed("fig7d-transmission-gamma") + k_s)
    pk, dif = [], []
    for _ in range(1000 if k_s == 0 else 200):
        mm = np.nanmean(isi[rng7d.integers(0, len(isi), len(isi))], 0)
        pk.append(int(np.nanargmax(mm))); dif.append(mm[5] - mm[4])
    pk, dif = np.array(pk), np.array(dif)
    print(f"seed +{k_s}: {len(pk)} resamplings; peak bin centre counts", {f"{centres[j]:.1f} ms": int((pk == j).sum()) for j in np.unique(pk)},
          f"; fraction with the peak in 13.5-25.3 ms {np.mean(inside[pk]):.3f}; 29-42 minus 20-29 ms bin mean {np.mean(dif):.5f}, 95% interval {np.percentile(dif, 2.5):.5f} to {np.percentile(dif, 97.5):.5f}")


**Result: not reproduced, resting on the assumed connection detector.** Mean transmission probability rises from 0.0190 at 5-7 ms ISIs to a peak of 0.0269 in the 29-42 ms bin (centre 34.9 ms), outside 13.5 to 25.3 ms; the shortest-ISI bin is below the peak, as the claim's depression at short intervals says. The peak exceeds the 20-29 ms bin (0.0266, centre 24.5 ms, inside the range) by 0.0003; with the connections resampled, that difference has a 95% interval of 0.00017 to 0.00050 and the peak is in the 29-42 ms bin in 999 of 1,000 resamplings, never inside 13.5 to 25.3 ms, under all five seeds. The peak stays in the same bin under every detector variant.

> Claim fig7fg-convergence-cofiring: Of 184,230 possible pyramidal-interneuron pairs, 31,932 are monosynaptically
> connected; 65.27% of pyramidal pairs share at least one postsynaptic interneuron, and cofiring in SPW-Rs, theta
> cycles and ratemap overlap all increase with convergence.

In [ ]:
RMAT = {r[2]["session"]: r[2]["rip_corr_mat"] for r in hc_res}
MAPV = {}
for x in mz:
    MAPV[x["session"]] = np.concatenate([x["maps"][0][:, STEM], x["maps"][1][:, STEM]], axis=1)
rows, nz, tot, anyt = [], 0, 0, 0
for x in mo:
    c = x["conn"].astype(int); n = len(c)
    if n < 2: continue
    sh = c @ c.T; un = c.shape[1] - (1 - c) @ (1 - c).T  # shared targets; union of targets
    iu = np.triu_indices(n, 1)
    nz += int((sh[iu] > 0).sum()); tot += len(iu[0]); anyt += int((un[iu] > 0).sum())
    lr = x["lr"]; rm = RMAT[x["session"]]; tm = TMAT[x["session"]]; V = MAPV.get(x["session"])
    for a, b in zip(*iu):
        if sh[a, b] == 0: continue
        rows.append({"session": x["session"], "nr_pair": (not lr[a]) and (not lr[b]), "cv": sh[a, b] / un[a, b],
                     "rip": rm[a, b], "theta": tm[a, b], "map": corr_visited(V[a], V[b]) if V is not None else np.nan})
Q = pd.DataFrame(rows)
nconn = int(sum(x["conn"].sum() for x in mo))
print("legend's pyramidal pairs (108,468 with zero and 207,254 with nonzero convergence):", 108468 + 207254, "; here, all pairs", tot, "and pairs with any target", anyt)
print(f"connections {nconn} of {npairs} possible; pyramidal pairs with >= 1 shared interneuron {nz} of {tot} ({100*nz/tot:.2f}%); "
      f"of pairs with any target {anyt} ({100*nz/max(anyt,1):.2f}%)")
def quint_reg(df, col, claim, k):
    d = df.dropna(subset=[col]).copy()
    d["q"] = pd.qcut(d.cv.rank(method="first"), 5, labels=False) + 1
    X = d.q.values.astype(float); Y = d[col].values
    def tstat(x, y):
        xm = x - x.mean(); b = (xm @ (y - y.mean())) / (xm @ xm)
        res = y - y.mean() - b * xm; se = np.sqrt((res @ res) / (len(y) - 2) / (xm @ xm)); return b, b / se
    b, t = tstat(X, Y)
    rng = np.random.default_rng(seed(claim) + k)
    null = np.array([tstat(X, rng.choice(Y, len(Y)))[1] for _ in range(1000)])
    return b, t, float(np.mean(np.abs(null) >= abs(t))), [round(float(v), 4) for v in d.groupby("q")[col].mean()], len(d)
for lab, sel in (("(a) nonresponsive pairs", Q.nr_pair), ("(b) all pyramidal pairs", Q.index >= 0)):
    for col in ("rip", "theta", "map"):
        b, t, p, qm, n = quint_reg(Q[sel], col, "fig7fg-convergence-cofiring", 0)
        print(f"{lab}, {col}: n = {n}, quintile means {qm}, slope {b:.4f}, t = {t:.2f}, bootstrap p = {p:.3f}")
for k in SEEDS_PLUS[1:]:
    print(f"seed +{k} (a):", [round(quint_reg(Q[Q.nr_pair], col, 'fig7fg-convergence-cofiring', k)[2], 3) for col in ("rip", "theta", "map")])
b, t, p, qm, n = quint_reg(Q[Q.nr_pair & Q.session.isin(RULE_SESS)], "theta", "fig7fg-convergence-cofiring", 0)
print(f"variant, (a) theta on the sessions whose channel follows the rule: n = {n}, quintile means {qm}, slope {b:.4f}, t = {t:.2f}, bootstrap p = {p:.3f}")
b, t, p, qm, n = quint_reg(Q[Q.nr_pair & ~Q.session.isin(RULE_SESS)], "theta", "fig7fg-convergence-cofiring", 0)
print(f"variant, (a) theta on the other sessions: n = {n}, quintile means {qm}, slope {b:.4f}, t = {t:.2f}, bootstrap p = {p:.3f}")


**Result: not reproduced, resting on the assumed connection detector.** The possible pairs are 184,230, the paper's number exactly, but the detector finds 50,618 connections (31,932 +- 3,193 expected) and 86.2% of pyramidal pairs share an interneuron (65.27 +- 6.53% expected). The slopes of SPW-R cofiring, theta cofiring and ratemap correlation on convergence quintile are positive under both readings (0.0040, 0.0028 and 0.0017 for nonresponsive pairs, reading a; 0.0042, 0.0029 and 0.0023 for all pairs, reading b; bootstrap p < 0.001 under all seeds), so the slope part holds. SPW-R and theta cofiring rise across all five quintiles; the ratemap correlation of nonresponsive pairs is flat over the first four (0.0111, 0.0115, 0.0109, 0.0096) and rises in the top quintile (0.0206). On the sessions whose theta channel follows the rule the theta slope is 0.0026 (p < 0.001). The detector variants give 38,402 to 50,756 connections and 74.5 to 86.4% of pairs with a shared interneuron, all outside tolerance. The legend's zero and nonzero pairs sum to 315,722, against 371,756 pyramidal pairs here (370,837 with any target), so the paper's pair set differs in a way the outputs do not show.

## Blocked claims

> Claim ed2-waveform-validation: Light-evoked spikes most resemble the spontaneous spikes of their own cluster, and
> light-responsive and nonresponsive pyramidal cells have the same waveform trough-to-peak.

The waveform distance needs the waveforms of light-evoked spikes, which need the wide-band signal around the
pulses. The processed files hold only each unit's mean waveform over all spikes; the wide-band signal is in the
11 raw files, of which 5 are homecage-mouse sessions (52 to 141 GB each). The claim is blocked. The
trough-to-peak part alone is printed here for the record; it does not decide the outcome.

In [ ]:
rows = []
for m in metas:
    if m["subject_id"] in h.HOMECAGE_EXCLUDED: continue
    for c, l, v in zip(m["u_putative_cell_type"], m["u_light_responsive"], m["u_trough_to_peak"]):
        if c == "Pyramidal Cell": rows.append({"lr": l, "ttp": v})
TT = pd.DataFrame(rows)
a, b = TT[TT.lr].ttp.dropna(), TT[~TT.lr].ttp.dropna()
print(f"trough-to-peak: light-responsive mean {a.mean():.3f} ms (n = {len(a)}), nonresponsive mean {b.mean():.3f} ms (n = {len(b)}), "
      f"two-sided t-test p = {stats.ttest_ind(a, b).pvalue:.3g}")

> Claim ed5-matched-controls: The SBD advantage in SPW-R, theta, familiar ratemap and novel ratemap correlations
> survives matching DBD pairs to SBD pairs in firing rate and in cluster isolation (L-ratio).

The L-ratio half cannot be run: the units table holds no isolation metric (data check). The rate-matched half is run
here for the record; it does not decide the outcome. DBD pairs are drawn to the SBD pairs' joint distribution of the two
cells' rates (`h.subsample2d`, subsample2D_local.m), to the SBD group size, 500 times. The rate used: SPW-R cofiring, the
SPW-R firing rate (legend); novel ratemap correlations, the theta-related rate in that novel environment (legend; wake
cycles of the theta section); theta cofiring and familiar ratemap correlations, the homecage firing rate (assumption: the
theta-related rate is computed here only for the novel sessions).

In [ ]:
rng5 = np.random.default_rng(seed("ed5-matched-controls"))
CI = CELLS.set_index(["session", "unit"])
def matched_fraction(df, col, va, vb, n_res=500):
    df = df.dropna(subset=[col])
    vals = np.c_[va(df), vb(df)]
    ok = np.all(np.isfinite(vals), 1); df, vals = df[ok], vals[ok]
    s_, d_ = df.sbd.values, ~df.sbd.values
    sv, dv, dc = vals[s_], vals[d_], df[col].values[d_]
    sc = df[col].values[s_]
    ps, meds = [], []
    for _ in range(n_res):
        k = h.subsample2d(sv, dv, len(sv), rng5)
        ps.append(stats.ranksums(sc, dc[k]).pvalue); meds.append(np.median(dc[k]))
    ps = np.array(ps)
    print(f"  {col}: SBD n {len(sc)} median {np.median(sc):.4f}; matched DBD median {np.median(meds):.4f}; fraction with p < 0.05 {np.mean(ps < 0.05):.3f}; "
          f"rate medians SBD {np.round(np.median(sv, 0), 3).tolist()} DBD {np.round(np.median(dv, 0), 3).tolist()}")
    return float(np.mean(ps < 0.05))
rr = lambda df, u: np.array([CI.rip_rate.get((s_, x_), np.nan) for s_, x_ in zip(df.session, df[u])])
hr = lambda df, u: np.array([CI.rate.get((s_, x_), np.nan) for s_, x_ in zip(df.session, df[u])])
f5 = {}
f5["spwr"] = matched_fraction(PAIRS, "rip_corr", lambda d: rr(d, "a"), lambda d: rr(d, "b"))
f5["theta"] = matched_fraction(TPH, "theta_corr", lambda d: hr(d, "a"), lambda d: hr(d, "b"))
f5["familiar"] = matched_fraction(MP, "r", lambda d: hr(d, "ua"), lambda d: hr(d, "ub"))
PYRMAP = {x["session"]: x["pyr"] for x in nv}
NR_ = TR[TR.wake_only & (TR.epoch > 0)].set_index(["session", "unit", "epoch"]).rate
NPn = NP[NP.epoch > 0].copy()
NPn["ua"] = [int(PYRMAP[s_][a_]) for s_, a_ in zip(NPn.session, NPn.a)]; NPn["ub"] = [int(PYRMAP[s_][b_]) for s_, b_ in zip(NPn.session, NPn.b)]
tr_ = lambda d, u: np.array([NR_.get((s_, x_, e_), np.nan) for s_, x_, e_ in zip(d.session, d[u], d.epoch)])
f5["novel"] = matched_fraction(NPn, "r", lambda d: tr_(d, "ua"), lambda d: tr_(d, "ub"))
print("rate-matched fractions:", {k: round(v, 3) for k, v in f5.items()}, "; paper: every resampling for SPW-R and theta, 499/500 familiar, 0.958 novel")

> Claim ed10-waveform-matched-controls: Same-shank SBD pairs have more similar waveforms than DBD pairs, but among
> same-shank pairs more than 60 micrometres apart, whose waveform distances do not differ, SBD pairs still cofire more in
> SPW-Rs and theta cycles and have higher familiar ratemap correlations.

The claim depends on ed2-waveform-validation, which is blocked, so it is blocked too. Its own parts need only the
spontaneous mean waveforms and the electrode layout, which the files hold (`waveform_mean` over all channels; data check),
so they are computed here for the record. Distance between two cells is that between their maximum-waveform channels;
the waveform profile distance is the L2 norm of the difference of the mean waveforms concatenated over the shank's good
channels (legend; the code's division by the number of channels as a variant). Waveform-matched resampling draws DBD
pairs to the SBD waveform-distance distribution (100 bins of log10 distance, as `h.subsample2d`), at the size of the
full same-shank SBD group, 500 times.

In [ ]:
rng10 = np.random.default_rng(seed("ed10-waveform-matched-controls"))
SS = PAIRS[PAIRS.shank_a == PAIRS.shank_b].copy()
far = SS[SS.el_dist > 60]
for lab, df in (("same shank", SS), ("same shank, > 60 um", far)):
    for col in ("wf_dist", "wf_dist_code"):
        a_, b_ = df[df.sbd][col].dropna(), df[~df.sbd][col].dropna()
        print(f"{lab}, {col}: SBD median {a_.median():.4g} (n = {len(a_)}), DBD {b_.median():.4g} (n = {len(b_)}), rank-sum p = {stats.ranksums(a_, b_).pvalue:.3g}")
TPk = TPH.set_index(["session", "a", "b"]).theta_corr
far = far.assign(theta_corr=[TPk.get((s_, a_, b_), np.nan) for s_, a_, b_ in zip(far.session, far.a, far.b)])
SSt = SS.assign(theta_corr=[TPk.get((s_, a_, b_), np.nan) for s_, a_, b_ in zip(SS.session, SS.a, SS.b)])
WF = SS.set_index(["session", "a", "b"]).wf_dist
MPs = MP[MP.dshank == 0].copy()
MPs["wf_dist"] = [WF.get((s_, a_, b_), np.nan) for s_, a_, b_ in zip(MPs.session, MPs.ua, MPs.ub)]
farm = MPs[MPs.el_dist > 60]
exp10 = {"rip_corr": (0.037, 0.021), "theta_corr": (0.019, 0.0092), "r": (0.0079, -0.0308)}
for col, df in (("rip_corr", far), ("theta_corr", far), ("r", farm)):
    a_, b_ = df[df.sbd][col].dropna(), df[~df.sbd][col].dropna()
    e_ = exp10[col]
    print(f"> 60 um, {col}: SBD median {a_.median():.4f} (n = {len(a_)}) DBD {b_.median():.4f} (n = {len(b_)}), rank-sum p = {stats.ranksums(a_, b_).pvalue:.3g}; "
          f"within tolerance {abs(a_.median() - e_[0]) <= 0.1 * abs(e_[0])}, {abs(b_.median() - e_[1]) <= 0.1 * abs(e_[1])}")
def wf_matched(df, col, n_res=500):
    df = df.dropna(subset=[col, "wf_dist"]); s_ = df[df.sbd]; d_ = df[~df.sbd]
    ps = [stats.ranksums(s_[col], d_[col].values[h.subsample2d(s_.wf_dist.values, d_.wf_dist.values, len(s_), rng10)]).pvalue for _ in range(n_res)]
    return float(np.mean(np.array(ps) < 0.05))
print("waveform-matched fractions with p < 0.05, all same-shank pairs: SPW-R", wf_matched(SS, "rip_corr"), "; theta", wf_matched(SSt, "theta_corr"),
      "; familiar ratemap", wf_matched(MPs, "r"), "(paper: every homecage resampling, 0.894 ratemap)")

> Claim ed7-sbd-coactivation: Groups of more than two SBD neurons coactivate in SPW-Rs more than equal-sized control
> groups; hard-coded SBD assemblies are expressed above control in 62 of 65 sessions.

The claim depends on fig3e-assembly-expression, which is blocked, so it is blocked too, and its part (b) needs the assembly
expression on the 1 ms matrix of that claim. Part (e) needs only which cells fire in each SPW-R, and is computed here for
the record: per session with at least two light-responsive cells, the fraction of homecage SPW-Rs in which exactly k of
them fire, z-scored against 1,000 groups of as many nonresponsive pyramidal cells drawn at random.

In [ ]:
rng7 = np.random.default_rng(seed("ed7-sbd-coactivation"))
zr = []
for r in hc_res:
    info = r[2]; lrc, nrc = info["lr_cols"], info["nr_cols"]
    if len(lrc) < 2 or len(nrc) < len(lrc) or info["n_rip_rows"] == 0: continue
    act = np.unpackbits(info["active"], axis=0, count=info["n_rip_rows"]).astype(bool)
    g = len(lrc); kk = np.arange(1, g + 1)
    obs = np.bincount(act[:, lrc].sum(1), minlength=g + 1)[1:] / len(act)
    sel = np.zeros((len(nrc), 1000), np.float32)
    for j in range(1000): sel[rng7.choice(len(nrc), g, replace=False), j] = 1
    cnt = (act[:, nrc].astype(np.float32) @ sel).astype(int)   # coactive cells of each random group in each SPW-R
    null = np.stack([(cnt == k).mean(0) for k in kk], axis=1)
    z = (obs - null.mean(0)) / np.where(null.std(0) > 0, null.std(0), np.nan)
    for k, zz in zip(kk, z): zr.append({"session": info["session"], "n_lr": g, "k": k, "z": zz})
Z7 = pd.DataFrame(zr)
print("sessions with >= 2 light-responsive cells:", Z7.session.nunique(), "(paper 84); with >= 3:", Z7[Z7.n_lr >= 3].session.nunique(), "(paper 65)")
print("mean z by k (finite values):", Z7.groupby("k").z.apply(lambda v: round(v[np.isfinite(v)].mean(), 3)).head(8).to_dict())
zz = Z7[(Z7.k > 2) & np.isfinite(Z7.z)].z
print(f"mean z for k > 2: {zz.mean():.3f} over {len(zz)} session-k values; one-sample t p = {stats.ttest_1samp(zz, 0).pvalue:.3g}")

The other blocked claims are not computed here: the histology of ed1-brdu-specificity and fig1c-radial-depth is
not in the dataset; the assembly claims (fig3c, fig3e, ed6) and the model of fig4d were not run (see the attempt record).

The resampled fractions of ed5 and ed10 come from one generator per claim shared across their comparisons, so a
change earlier in the sequence (here the theta pairs, after the theta channel was revised) moves the later ones: the
novel ratemap fraction of ed5 was 0.604 in the previous run and is 0.498 here, and the ratemap fraction of ed10 0.106
and 0.132.

In [ ]:
print(f"notebook runtime {(time.time() - T0)/60:.1f} min; peak memory of this process {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1e6:.2f} GB")

## Summary

| Claim | Outcome |
|---|---|
| cohort-birthdates | reproduced |
| st1-animal-sessions | reproduced |
| sessions-durations | not reproduced (two sessions without sleep; all else holds; asked of a reader) |
| spwr-duration | reproduced |
| unit-count | reproduced |
| ed2-light-responsive-count | reproduced |
| ed2-waveform-validation | blocked: data missing |
| ed1-brdu-specificity, fig1c-radial-depth | blocked: data missing |
| fig1e-labelled-fraction | reproduced |
| fig1f-rem-shift | not reproduced (assumed theta channel) |
| fig1g-rate-by-birthdate | reproduced |
| fig1h-burst-by-birthdate | partially reproduced |
| ed3-depth-control | not reproduced (fractions above the paper's; assumed depth reference) |
| fig2b-synchrony-per-animal | reproduced |
| fig2c-theta-cofiring | partially reproduced (assumed theta channel) |
| fig2d-theta-depth | reproduced (assumed theta channel) |
| fig2f-ripple-firing-by-birthdate | reproduced |
| fig2g-ripple-cofiring | reproduced |
| ed5-matched-controls | blocked: data missing (L-ratio) |
| fig3c, fig3e, ed6, fig4d | blocked: too large (not run; see the attempt record) |
| ed7-sbd-coactivation | blocked: too large (depends on fig3e; part e run for the record) |
| maze-cohort | reproduced |
| maze-task | not reproduced (trial counts) |
| novel-explorations | reproduced |
| fig5b-spatial-info-by-birthdate | partially reproduced |
| fig5c-sbd-ratemap-overlap | partially reproduced |
| fig5d-overlap-by-distance | partially reproduced |
| fig5e-pv-decorrelation | partially reproduced |
| fig5f-remap-together | reproduced |
| fig6c-theta-rate-preserved | reproduced (assumed theta channel) |
| fig6d-rate-lower-novel | reproduced (assumed theta channel) |
| fig6d-info-lower-novel | partially reproduced |
| fig6c-novel-ratemap-corr-higher | not reproduced (assumed open-field scale) |
| fig6e-sbd-novel-overlap | partially reproduced |
| fig6f-remap-together-novel | reproduced (assumed open-field scale) |
| fig7b-sbd-convergence | partially reproduced (assumed detector) |
| fig7ce-convergence-by-birthdate | partially reproduced (assumed detector) |
| fig7d-transmission-gamma | not reproduced (assumed detector) |
| fig7fg-convergence-cofiring | not reproduced (assumed detector) |
| ed10-waveform-matched-controls | blocked: data missing (depends on ed2-waveform-validation) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the
differences from the paper's method.

How the analysis differs from the paper's. The theta channel follows the Methods' rule (a channel with a positive
sharp wave, measured against a pre-ripple baseline, above the centre of the pyramidal layer) in 72 of 104 sessions; in
the other 32 no shank has a channel above its centre with a positive sharp wave, and the channel next to the
centre with larger rel_y is used. Six sessions with 20 or fewer ripples in homecage non-REM sleep take the sharp wave over all their
ripples. Every theta claim is also shown on the sessions whose channel follows the rule, and every one but
fig7fg-convergence-cofiring on the layer centre channel. The monosynaptic detector follows the Methods' description with assumed lags, kernel, Bonferroni count and
minimums, and finds 38,402 to 50,756 connections under its variants, 20% to 59% more than the paper's 31,932. Open
fields are binned assuming one lab unit is 1 cm, with 2, 0.5 and 0.85 cm per unit as variants. Depth for the depth
control is taken from the sharp-wave reversal of each shank face, and from the ripple-power centre where no reversal is
found. Rate maps use a constant-velocity Kalman filter whose settings the paper does not give. The bootstrap post hoc
test is replaced by Tukey's test on the six distance and birthdate groups.

What was found about the dataset. The files give the paper's counts of mice, sessions, units, cell types
and light-responsive cells exactly, and the paper's per-session labelled fractions. Two sessions of e13_1f1 hold
wake only in their sleep states. In e16_3m2_211211 the position series named for the linear_manu
subsession lies inside the openField_thomas subsession. Open-field positions have no spatial scale, and the
lab-unit positions of the other novel environments use different scales (0 to 1 in five series, 0.85 to 0.93 cm per
unit in three, from their simultaneous linearized positions). The units table has no L-ratio. Mice e13_1f1 and e13_1m1, which the paper says had a single
light-responsive cell per session, have 1 and 2 same-birthdate pairs in the files. 27 of 57 familiar-maze
sessions hold fewer than ten trials of one type, in the trials table itself.